# Tactical Systems for Hex Strategy

Comprehensive tactical AI and algorithms for hex-based warfare and logistics.

## Overview

This document catalogs tactical patterns, formation algorithms, and strategic concepts for coordinating multiple pieces on a hex grid. Each tactic is backed by algorithms from graph theory, optimization, and computational geometry.

---

## Implemented Tactics

### 1. Surround (FEED/GUARD Modes)

**Purpose:** Position pieces around a high-value target (queen, settlement, chokepoint).

**Algorithm:** Hungarian matching (linear sum assignment)

```python
surround(pieces, target, grid, elevations,
         mode=SurroundMode.GUARD,  # or FEED
         max_ring=3)
```

**Implementation:**
1. Generate candidate positions (rings 1..max_ring around target)
2. Build cost matrix: `cost[piece, position] = pathfind_cost(piece → position)`
3. Constraint: `effective_sight(piece) >= distance(position, target)`
4. `linear_sum_assignment(cost)` → optimal piece↔position pairing
5. Generate paths + final facing (inward for FEED, outward for GUARD)

**Modes:**
- **FEED** — Face inward, supply chain formation, GIVE cones converge on center
- **GUARD** — Face outward, defensive perimeter, sight cones scan threats

**Complexity:** O(n³) for Hungarian, O(nm log m) for pathfinding (n=pieces, m=hexes)

**Strategic Uses:**
- Protect high-value piece (GUARD queen while she harvests)
- Establish supply hub (FEED formation feeds queen who distributes)
- Hold chokepoint (GUARD formation watches all approaches)

---

## Proposed Tactics

### 2. Pincer / Flanking

**Purpose:** Attack from multiple directions to divide enemy attention and cut retreat.

**Algorithm:** Two-group surround with asymmetric targeting

**Concept:**
- Split attackers into left and right wings
- Each wing surrounds one flank of target's position
- Wings converge from opposite sides (±90° from direct approach)

**Implementation:**
```python
def pincer(pieces, target, grid, elevations,
           flank_angle=90,    # degrees from center line
           attack_rings=2):   # how close to get
    """
    Flank from both sides.
    
    1. Split pieces into left/right wings (balanced by strength)
    2. Generate left-flank positions (target + offset in left direction)
    3. Generate right-flank positions (target + offset in right direction)
    4. Hungarian match each wing to its positions
    5. All pieces face toward target for coordinated assault
    """
```

**Algorithm Details:**
1. Compute center line from center-of-mass of pieces to target
2. Rotate ±`flank_angle` degrees to get left/right approach vectors
3. Generate positions along arcs at `attack_rings` distance from target
4. Partition pieces: high-attack units to wings, support units to center
5. Two independent Hungarian problems (left wing, right wing)
6. Synchronize movement so wings arrive simultaneously

**Strategic Uses:**
- Overwhelm defensive formation
- Cut off retreat path
- Force enemy to split attention between two fronts

---

### 3. High Ground Control

**Purpose:** Claim elevated positions for sight/defense advantage.

**Algorithm:** Elevation-weighted greedy assignment

**Concept:**
- Elevation grants sight bonus (`sight += elevation × 0.005`)
- Higher hexes see farther → better for archers, scouts, GIVE range
- Defensive advantage (enemy climbs uphill → slower, visible)

**Implementation:**
```python
def seize_high_ground(pieces, region: HexRegion, grid, elevations):
    """
    Assign pieces to high-elevation hexes in region.
    
    1. Rank all hexes in region by elevation
    2. Take top N hexes (N = number of pieces)
    3. Build cost matrix: pathfind cost + elevation penalty
       cost[i,j] = path_cost - elevation[j] * ELEV_WEIGHT
    4. Hungarian match pieces to high ground
    5. Generate paths, face outward for visibility
    """
```

**Scoring Function:**
```python
score(hex) = elevation * ELEV_WEIGHT - path_cost
```

**Strategic Uses:**
- Scouting — see enemy movements from peaks
- Artillery position — long-range pieces get max effective sight
- Defensive strongpoint — force enemy to attack uphill

---

### 4. Supply Line / Relay Chain

**Purpose:** Establish GIVE chain from harvesters to frontline.

**Algorithm:** Steiner tree approximation + flow optimization

**Concept:**
- Harvesters at food sources GIVE to relays
- Relays GIVE to frontline consumers
- Minimize total path length while ensuring connectivity

**Implementation:**
```python
def supply_chain(harvesters, consumers, grid, elevations, food_tiers):
    """
    Build relay chain connecting harvesters → consumers.
    
    1. Find high-yield food hexes (food_tier >= threshold)
    2. Assign harvesters to food hexes (nearest-match)
    3. Compute Steiner tree connecting harvesters + consumers
    4. Place relays at Steiner vertices
    5. Orient each piece's GIVE cone toward next hop in chain
    """
```

**Steiner Tree Approximation:**
1. Build minimum spanning tree (MST) on (harvesters ∪ consumers)
2. Use Dijkstra to compute shortest paths between all pairs
3. MST edges become supply routes
4. Relays positioned at branch points

**Flow Constraints:**
- Each relay must have `food_capacity >= downstream_demand`
- GIVE cones must cover next hop (facing check)
- Reserve `4 × diet` for self

**Strategic Uses:**
- Sustain prolonged siege
- Feed distant expansion force
- Enable low-harvest pieces (knights) to stay at front

---

### 5. Patrol Routes (Gosper Curve)

**Purpose:** Efficiently cover area without backtracking.

**Algorithm:** Space-filling curve generation

**Concept:**
- Gosper curve is hex-native space-filling curve
- Covers region with minimal repeated hexes
- Natural for patrol, scouting, area denial

**Implementation:**
```python
def patrol_route(anchor: int, radius: int, grid: HexGrid,
                 curve_order=2) -> list[int]:
    """
    Generate Gosper curve patrol route.
    
    1. Build Gosper curve of order N around anchor
    2. Filter to hexes within radius
    3. Convert curve to hex indices
    4. Return as InstructionList (path_to_rules)
    """
```

**Patrol Behaviors:**
- **SCOUT** — high sight pieces on wide patrol (order 3-4)
- **PERIMETER** — patrol kingdom border (anchor = capital)
- **SWEEP** — search for enemy in region

**Strategic Uses:**
- Early game exploration
- Border security
- Resource scouting

---

### 6. Follow the Leader

**Purpose:** Move formation relative to leader's position.

**Algorithm:** Offset maintenance

**Concept:**
- Designate one piece as leader
- Other pieces maintain fixed offset from leader
- Formation moves as a rigid body

**Implementation:**
```python
def follow_formation(leader: Piece, followers: list[Piece],
                     grid: HexGrid, offsets: dict[str, HexPosition]):
    """
    Maintain formation relative to leader.
    
    Args:
        offsets: piece.id → desired HexPosition offset from leader
    
    Each turn:
    1. Compute target = leader.location + offset
    2. If piece not at target: pathfind_to_rules(target)
    3. Face same direction as leader
    """
```

**Formation Templates:**
- **Line** — pieces in row behind leader
- **Wedge** — V-shape with leader at point
- **Box** — surround leader at ring-1
- **Column** — single file (for narrow passages)

**Strategic Uses:**
- Advance through difficult terrain as unit
- Retreat without breaking formation
- Convoy protection (civilians surrounded by guards)

---

### 7. Concentration of Force

**Purpose:** Mass pieces at single point for overwhelming attack.

**Algorithm:** Convergent pathfinding with timing

**Concept:**
- All pieces path to same target hex
- Arrivals synchronized (slower pieces start earlier)
- Coordinated assault on specific turn

**Implementation:**
```python
def converge(pieces, target, grid, elevations, assault_turn):
    """
    Coordinate multi-turn convergence.
    
    1. For each piece: pathfind to target
    2. Compute arrival_turn = current_turn + path_length / move_speed
    3. If arrival_turn < assault_turn: add PAUSE instructions
    4. If arrival_turn > assault_turn: piece too slow, assign alternate target
    5. All pieces execute simultaneously
    """
```

**Timing Constraint:**
```python
start_turn[piece] = assault_turn - path_length / move_strength
```

**Strategic Uses:**
- Coordinated siege
- Overwhelm isolated enemy
- Capture key objective (e.g., settlement)

---

### 8. Leapfrog Advance

**Purpose:** Alternating waves advance while covering each other.

**Algorithm:** Staged movement with overwatch

**Concept:**
- Group A advances while Group B provides sight/defense
- Group B advances while Group A sets up
- Repeat, trading roles each turn

**Implementation:**
```python
def leapfrog(vanguard: list[Piece], rearguard: list[Piece],
             destination: int, grid, elevations):
    """
    Alternating advance.
    
    Turn 1: vanguard moves forward, rearguard stays (DEFEND/GIVE)
    Turn 2: rearguard moves past vanguard, vanguard stays
    Repeat until both groups reach destination
    """
```

**Variant: Bounding Overwatch**
- Rearguard always has line-of-sight to vanguard
- GIVE cones cover vanguard's advance
- Rearguard includes high-sight pieces

**Strategic Uses:**
- Advance through contested territory
- Maintain supply chain during movement
- Minimize exposure to ambush

---

### 9. Defensive Line / Phalanx

**Purpose:** Hold position against frontal assault.

**Algorithm:** Line formation with mutual support

**Concept:**
- Pieces form contiguous line
- Each piece covers neighbors with GIVE/sight
- High-defense pieces at front, support at back

**Implementation:**
```python
def phalanx(pieces, line_start, line_end, grid, elevations):
    """
    Form defensive line.
    
    1. Generate hex positions along line from start to end
    2. Sort pieces: high HP front, high harvest_strength back
    3. Hungarian match pieces to line positions
    4. Face perpendicular to line (outward)
    5. GIVE cones point along line for mutual support
    """
```

**Line Generation:**
- Bresenham-like hex line algorithm
- Thicken to 2-3 hexes deep if enough pieces

**Variants:**
- **Shield Wall** — all pieces face same direction (enemy approach)
- **Hedgehog** — pieces face alternating directions (all-around defense)

**Strategic Uses:**
- Hold chokepoint (valley, bridge)
- Defend settlement perimeter
- Block enemy advance

---

### 10. Feint / Bait

**Purpose:** Lure enemy out of position with fake attack.

**Algorithm:** Pathfinding with retreat trigger

**Concept:**
- Small force advances toward enemy
- When enemy responds, retreat to prepared position
- Main force attacks exposed flank

**Implementation:**
```python
def feint(bait_pieces, main_force, enemy_position,
          grid, elevations, retreat_distance=5):
    """
    Lure enemy with sacrifice pieces.
    
    Bait group:
    1. Advance toward enemy (2/3 distance)
    2. If enemy within sight: retreat to rally point
    3. Rally point = starting position + retreat_distance
    
    Main force:
    1. Position at flanking angle from rally point
    2. Wait for bait to pull enemy
    3. Attack enemy flank when exposed
    """
```

**Trigger Condition:**
```python
if any(enemy in bait.pieces_in_sight(...)):
    execute_retreat()
```

**Strategic Uses:**
- Draw enemy from fortified position
- Split enemy formation
- Exhaust enemy movement (chasing bait)

---

## Advanced Tactical Concepts

### 11. Envelopment

**Purpose:** Surround enemy from all sides, cut off retreat.

**Algorithm:** Multi-group surround with rear blocker

**Concept:**
- Combine flanking (sides) + rearguard block (escape route)
- Three groups: left, right, rear
- Front group pins enemy while sides/rear close

**Implementation:**
3 simultaneous Hungarian problems:
1. Left wing → left flank positions
2. Right wing → right flank positions  
3. Rear group → positions behind enemy

**Historical Example:** Cannae (Hannibal)

---

### 12. Refuse Flank

**Purpose:** Concentrate force on one flank while refusing other.

**Algorithm:** Asymmetric deployment

**Concept:**
- Strong wing advances (attack)
- Weak wing retreats slowly (economy of force)
- Enemy's advance on weak wing overextends

**Implementation:**
```python
strong_wing: attack_positions (close)
weak_wing:   defensive_positions (far back)
```

**Historical Example:** Leuthen (Frederick the Great)

---

### 13. Hammer and Anvil

**Purpose:** Fix enemy against obstacle while attacking from mobility.

**Algorithm:** Pin + crush

**Concept:**
- Anvil: immobile force holds enemy in place
- Hammer: mobile force attacks from flank/rear
- Enemy crushed between two forces

**Implementation:**
```python
anvil_pieces: phalanx(enemy_front)
hammer_pieces: pincer(enemy_rear)
```

---

### 14. Interior Lines

**Purpose:** Use central position to defeat enemies in detail.

**Algorithm:** Sequential force concentration

**Concept:**
- Friendly forces centrally positioned
- Enemies approaching from multiple directions
- Defeat each enemy force before others arrive

**Implementation:**
```python
for enemy_group in sorted_by_arrival_time:
    converge(all_pieces, enemy_group.location)
    defeat(enemy_group)
    reposition_to_center()
```

---

### 15. Withdrawal / Fighting Retreat

**Purpose:** Preserve force while breaking contact.

**Algorithm:** Phased retreat with rearguard

**Concept:**
- Main body retreats
- Rearguard delays pursuers
- Rearguard leapfrogs through main body when pressed

**Implementation:**
```python
while distance(main_body, enemy) < safe_distance:
    main_body.move_away()
    rearguard.defend()
    if rearguard_threatened:
        rearguard.leap_through_main_body()
```

---

## Algorithms Catalog

Beyond the tactics above, here are useful algorithms for strategy games:

### Graph Algorithms

#### 1. **Dijkstra's Algorithm** ✅ (Implemented)
- **Use:** Shortest path with weighted edges
- **Application:** Pathfinding with elevation costs
- **Complexity:** O((V+E) log V) with priority queue

#### 2. **A* Search**
- **Use:** Heuristic-guided shortest path
- **Application:** Faster pathfinding when destination known
- **Heuristic:** Euclidean distance (hex coords)
- **Complexity:** O(E log V) typical, O(V log V) worst

#### 3. **Bidirectional Search**
- **Use:** Meet-in-the-middle pathfinding
- **Application:** Long-distance paths, reduces search space
- **Complexity:** O(√V) instead of O(V)

#### 4. **Jump Point Search (JPS)**
- **Use:** Fast pathfinding on uniform grids
- **Application:** Large maps with few obstacles
- **Caveat:** Requires adaptation for hex grids
- **Complexity:** O(E log V) but with smaller constant

#### 5. **Bellman-Ford Algorithm**
- **Use:** Shortest paths with negative edges
- **Application:** Modeling bonuses (friendly territory = negative cost)
- **Complexity:** O(VE)

#### 6. **Floyd-Warshall Algorithm**
- **Use:** All-pairs shortest paths
- **Application:** Precompute distance matrix for small maps
- **Complexity:** O(V³)

#### 7. **Minimum Spanning Tree (MST)** - Kruskal/Prim
- **Use:** Connect nodes with minimum total edge weight
- **Application:** Supply line network, road building
- **Complexity:** O(E log V)

#### 8. **Steiner Tree Approximation**
- **Use:** Connect terminals through intermediate points
- **Application:** Multi-drop supply routes (harvesters → relays → consumers)
- **Complexity:** NP-hard, 2-approximation in O(V³)

#### 9. **Maximum Flow (Ford-Fulkerson, Dinic)**
- **Use:** Find maximum throughput in network
- **Application:** Supply capacity limits, troop movement bandwidth
- **Complexity:** O(VE²) Ford-Fulkerson, O(V²E) Dinic

#### 10. **Minimum Cut (Max-Flow Min-Cut)**
- **Use:** Find weakest break in network
- **Application:** Identify chokepoints, vulnerable supply routes
- **Complexity:** Same as max-flow

#### 11. **Strongly Connected Components (Tarjan's)**
- **Use:** Find mutually reachable regions
- **Application:** Identify isolated territory, cut-off regions
- **Complexity:** O(V+E)

#### 12. **Topological Sort**
- **Use:** Order nodes in DAG
- **Application:** Tech tree dependencies, instruction sequencing
- **Complexity:** O(V+E)

---

### Optimization Algorithms

#### 13. **Hungarian Algorithm** ✅ (Implemented)
- **Use:** Minimum-cost bipartite matching
- **Application:** Assign pieces to positions optimally
- **Complexity:** O(n³)

#### 14. **Hopcroft-Karp Algorithm**
- **Use:** Maximum cardinality bipartite matching
- **Application:** Maximum piece assignments (ignoring cost)
- **Complexity:** O(E√V)

#### 15. **Linear Programming (Simplex)**
- **Use:** Optimize linear objective with constraints
- **Application:** Resource allocation, production planning
- **Complexity:** Exponential worst-case, polynomial average

#### 16. **Integer Linear Programming (ILP)**
- **Use:** LP with integer constraints
- **Application:** Discrete decisions (assign this piece? yes/no)
- **Complexity:** NP-hard, branch-and-bound practical

#### 17. **Dynamic Programming**
- **Use:** Optimal substructure problems
- **Application:** Knapsack (carry weight), longest common subsequence
- **Complexity:** Pseudo-polynomial, often O(nW) for knapsack

#### 18. **Greedy Algorithms**
- **Use:** Locally optimal choices
- **Application:** Huffman coding, activity selection
- **Example:** High-ground capture (pick highest hex iteratively)

---

### Geometric Algorithms

#### 19. **Convex Hull (Graham Scan)**
- **Use:** Smallest convex shape enclosing points
- **Application:** Identify territory boundary, detect encirclement
- **Complexity:** O(n log n)

#### 20. **Voronoi Diagram**
- **Use:** Partition plane by nearest-point regions
- **Application:** Territory influence, nearest-settlement zones
- **Complexity:** O(n log n)

#### 21. **Delaunay Triangulation**
- **Use:** Dual of Voronoi, maximizes minimum angle
- **Application:** Road network (edges), natural borders
- **Complexity:** O(n log n)

#### 22. **Line-of-Sight (Bresenham/DDA)**
- **Use:** Check visibility between two points
- **Application:** Can piece see target? (with terrain blocking)
- **Complexity:** O(distance)

#### 23. **Field-of-View (Shadow Casting)** ✅ (Implemented)
- **Use:** All visible hexes from viewpoint
- **Application:** `field_of_view()` for cone generation
- **Complexity:** O(sight²)

#### 24. **Coverage Sets** ✅ (Mentioned)
- **Use:** Minimum pieces to cover all hexes
- **Application:** Watchtower placement, minimum garrison
- **Complexity:** NP-hard (set cover), greedy approximation

---

### Computational Geometry

#### 25. **Point-in-Polygon**
- **Use:** Is hex inside region?
- **Application:** Territory checks, region queries
- **Complexity:** O(n) ray-casting

#### 26. **Polygon Clipping (Sutherland-Hodgman)**
- **Use:** Intersect two polygons
- **Application:** Overlapping territory, region intersection
- **Complexity:** O(n+m)

#### 27. **Sweep Line Algorithm**
- **Use:** Process events in spatial order
- **Application:** Intersection detection, range queries
- **Complexity:** O(n log n)

---

### Simulation & AI

#### 28. **Monte Carlo Tree Search (MCTS)**
- **Use:** Best move via random simulation
- **Application:** Strategy game AI, AlphaGo-style planning
- **Complexity:** Time-bounded

#### 29. **Minimax with Alpha-Beta Pruning**
- **Use:** Adversarial game tree search
- **Application:** Chess-like tactical decisions
- **Complexity:** O(b^d) branching^depth, pruning reduces

#### 30. **Influence Maps**
- **Use:** Heatmap of control/danger
- **Application:** Identify safe zones, contested areas
- **Complexity:** O(V) diffusion

#### 31. **Behavior Trees**
- **Use:** Hierarchical AI decision structure
- **Application:** Complex piece behaviors (if starving → seek food, else patrol)
- **Complexity:** O(tree depth)

---

### Heuristics & Approximations

#### 32. **Hill Climbing**
- **Use:** Local search optimization
- **Application:** Refine formation positions
- **Complexity:** O(iterations × neighbors)

#### 33. **Simulated Annealing**
- **Use:** Probabilistic local search (escapes local optima)
- **Application:** Large-scale unit positioning
- **Complexity:** O(iterations)

#### 34. **Genetic Algorithms**
- **Use:** Evolutionary optimization
- **Application:** Learn piece behavior rules
- **Complexity:** O(generations × population)

---

## Tactical Primitives

These are building blocks used in larger tactics:

### Positioning
- **Adjacent** — next to target (ring 1)
- **Surround** — all 6 neighbors occupied
- **Flank** — 90° from enemy facing
- **Rear** — 180° from enemy facing

### Movement
- **Advance** — toward enemy/objective
- **Retreat** — away from threat
- **Reposition** — lateral shift
- **Rally** — converge on point

### Facing
- **Face toward** — orient at target
- **Face away** — orient opposite
- **Face formation** — all same direction
- **Face perimeter** — outward from center

### Actions
- **Hold** — DEFEND, no movement
- **Support** — GIVE to nearby
- **Harvest** — gather resources
- **Scout** — maximize sight coverage

---

## Integration with Game Systems

### Food System Integration

All tactics must consider food logistics:

**Harvest Stops:**
```python
if piece.food < piece.diet * 4:  # 4-turn reserve
    # Interrupt tactical move to harvest
    detour_to_food()
```

**Supply Lines:**
```python
tactical_formation() + supply_chain()
# Ensure frontline pieces get food from rear
```

**Starvation Risk:**
- Long marches require harvesters or relays
- Sieges need established supply
- Retreats may abandon food sources

### Elevation System Integration

Tactics must account for terrain:

**High Ground Bonus:**
```python
effective_sight = piece.sight + elevation * 0.005
```

**Uphill Movement Penalty:**
```python
move_cost = base_cost * (1 + elevation_diff * 0.1)
```

**Tactical Implications:**
- Archers on peaks
- Cavalry in valleys
- Defend hilltops

---

## Implementation Priorities

### Phase 1: Core (Implemented) ✅
- [x] Surround (FEED/GUARD)
- [x] Pathfinding (Dijkstra)
- [x] Pieces in sight (FOV + elevation)
- [x] Food mechanics

### Phase 2: Essential
- [ ] High ground control
- [ ] Supply chain
- [ ] Pincer/flanking
- [ ] Patrol routes (Gosper)

### Phase 3: Advanced
- [ ] Leapfrog advance
- [ ] Phalanx/defensive line
- [ ] Concentration of force
- [ ] Follow-the-leader

### Phase 4: Expert
- [ ] Feint/bait
- [ ] Envelopment
- [ ] Interior lines
- [ ] Fighting retreat

---

## Testing & Visualization

### Tactical Overlays

For debugging and demonstration:

```python
class TacticalOverlay:
    """
    Visualize tactical assignments on map.
    
    - Color pieces by assignment (blue=wing1, red=wing2)
    - Draw arrows showing movement paths
    - Show facing cones
    - Highlight objectives (target hex)
    """
```

### Metrics

Evaluate tactic effectiveness:

- **Coverage** — % of region under friendly sight
- **Concentration** — avg distance between pieces
- **Exposure** — % of pieces visible to enemy
- **Supply** — % of pieces within GIVE range of food
- **Response Time** — turns to reach objective

---

## Future Directions

### Machine Learning

Train piece behaviors:
- Reinforcement learning for tactic selection
- Neural network for position evaluation
- Genetic algorithms for rule evolution

### Dynamic Tactics

Adapt to battlefield conditions:
- Switch from GUARD to FEED when starving
- Break phalanx to pursue fleeing enemy
- Abandon high ground if flanked

### Combined Arms

Coordinate piece types:
- Knights pin, archers strike
- Pawns harvest, bishops distribute
- Rooks block, queen commands

---

## References

### Classical Military Theory
- **Sun Tzu** — The Art of War (interior lines, deception)
- **Carl von Clausewitz** — On War (concentration of force)
- **Liddell Hart** — Strategy (indirect approach)

### Game AI
- **Steering Behaviors** — Craig Reynolds (flocking, separation)
- **GOAP** — Goal-Oriented Action Planning
- **HTN Planning** — Hierarchical Task Networks

### Algorithms
- **Introduction to Algorithms** — CLRS (graph algorithms)
- **Computational Geometry** — de Berg et al.
- **Combinatorial Optimization** — Papadimitriou & Steiglitz

---

## Summary Table

| Tactic | Algorithm | Complexity | Strategic Goal |
|--------|-----------|------------|----------------|
| Surround | Hungarian | O(n³) | Protect/contain |
| Pincer | Split Hungarian | O(n³) | Flank attack |
| High Ground | Greedy + Hungarian | O(n² log n) | Sight advantage |
| Supply Chain | Steiner Tree | O(V³) | Sustain force |
| Patrol | Gosper Curve | O(n) | Area coverage |
| Follow Leader | Offset maintain | O(n) | Formation move |
| Converge | Dijkstra + timing | O(nm log m) | Mass attack |
| Leapfrog | Staged movement | O(n) | Safe advance |
| Phalanx | Line assignment | O(n²) | Hold position |
| Feint | Path + trigger | O(n log n) | Lure enemy |

---

## Next Steps

1. **Implement Phase 2 tactics** (high ground, supply chain, pincer)
2. **Build tactical overlay system** for visualization
3. **Create tactic composer** — combine primitives into complex strategies
4. **Add A* pathfinding** for performance
5. **Test on real scenarios** (siege, retreat, resource contest)

The tactical system is designed to be modular — each tactic is independent but composable. As the game evolves, new tactics can be added by mixing these primitives and algorithms.


```markdown
# Maneuvers & Formations — API Reference

## Core Primitives

### TroopPath

A resolved hex route with cached metadata. Returned by all pathfinding functions.

```python
path = piece.pathfind(target_hex, grid, elevations, countries=None)

path.hexes       # list[int] — ordered hex indices
path.cost        # float — total movement cost
path.start       # first hex
path.end         # last hex
path.as_set      # set[int] — cached for fast membership checks
path.to_rules()  # list[int] — convert to Instruction values

bool(path)       # False if empty or single-hex (unreachable)
len(path)        # number of hexes in route
```

**Movement costs:** flat/downhill = 1.0, uphill = 2.0. Water and invalid hexes are impassable.

### Pathfinding Variants

```python
# Basic A→B shortest path (Dijkstra)
path = piece.pathfind(target_hex, grid, elevations, countries)

# Nearest hex with food_tier >= threshold
path = piece.pathfind_to_food(grid, elevations, food_tiers, min_tier=3, countries)

# Path AWAY from a threat — prefers high ground in the opposite hemisphere
path = retreat_path(piece, threat_hex, grid, elevations,
                    distance=5, countries=None, elevation_weight=0.5)

# Convenience: pathfind + convert to InstructionList in one call
il = piece.pathfind_to_rules(target, grid, elevations, countries)
```

### InstructionList

The rule program for a piece. Each rule is an `Instruction` enum value.

```python
il = InstructionList(rules=[...], cursor=0, patrol=False)

il.insert_rules(new_rules)   # insert at cursor, advance cursor
il.delete_at(idx)             # remove one rule, clamp cursor
il.set_cursor(idx)            # move cursor
il.toggle_patrol()            # flip loop mode
il.est_turns(speed)           # lower-bound turn count for sync
il.compact(piece_id)          # FastHTML badge strip
```

**Patrol mode:** when `patrol=True`, cursor wraps to 0 at end of rules.
When `patrol=False`, piece stops after the last instruction.

### Instruction Budget Rules

Per turn, each piece gets `move_strength` budget points:

| Instruction | Cost | Notes |
|-------------|------|-------|
| FORWARD | 1 (flat/downhill) or 2 (uphill) | Into water/edge: auto-rotate left free, up to 5×. Into occupied hex: forced PAUSE cost 1. |
| ROT_L, ROT_R | 0 (free) | |
| PAUSE | 1 | |
| DEFEND | 1 | |
| HARVEST | 1 | |
| GIVE | 1 | |
| SETTLE | 1 | |

When budget runs out mid-instruction, the turn ends. If the last instruction
was non-movement (PAUSE/DEFEND/HARVEST/GIVE/SETTLE), remaining budget is
idled out as implicit PAUSEs.

---

## Formations

### FormationType

```python
class FormationType(Enum):
    LINE   = "line"    # row abreast, one step behind leader
    WEDGE  = "wedge"   # V-shape, leader at point, wings fan back
    BOX    = "box"     # ring-1 around leader (overflow to ring-2 if >6)
    COLUMN = "column"  # single file directly behind leader
```

**Choosing a formation:**
- **COLUMN** — narrow passages, rivers, mountain paths
- **LINE** — broad front for harvesting or sight coverage
- **WEDGE** — aggressive advance, concentrates force at point of contact
- **BOX** — protect a VIP (queen, settler), all-around defense

### form_up

Assemble followers around the leader's **current position**.

```python
orders = squad.form_up(
    FormationType.WEDGE, grid, elevations,
    leader=None,          # auto-picks highest rank if omitted
    countries=None,
    food_tiers=None,      # enables harvest padding during sync
    sync=True,            # pad so all arrive same turn
)
# orders: {piece_id: InstructionList} — leader NOT included
```

**Important:** `form_up` does not move the leader. It positions followers
relative to where the leader already is. Compose with `march_to` or
`pathfind_to_rules` for the leader's own movement.

### march_to

Move entire squad to a destination, arriving in formation.

```python
orders = squad.march_to(
    destination_hex,
    FormationType.LINE, grid, elevations,
    leader=None,          # auto-picks highest rank
    countries=None,
    food_tiers=None,
    sync=True,
)
# orders: {piece_id: InstructionList} — leader IS included
```

Formation offsets are computed relative to **destination + leader's arrival
facing**, so the formation assembles correctly on arrival. Uses Hungarian
matching to optimally assign followers to slots.

---

## Tactical Maneuvers

### surround

Position squad members around a target hex in a ring.

```python
orders = squad.surround(
    target_hex, grid, elevations,
    mode=SurroundMode.GUARD,  # GUARD = face outward, FEED = face inward
    max_ring=3,
    countries=None,
    elevation_mult=0.005,     # sight bonus per elevation unit
)
```

**Modes:**
- `GUARD` — defensive perimeter; sight cones scan outward for threats
- `FEED` — supply hub; GIVE cones converge on center piece

Uses Hungarian matching. Respects `effective_sight` — pieces won't be placed
farther than they can see.

### retreat

Scatter squad away from a threat. Each piece picks its own best retreat hex
independently (no formation maintained during withdrawal).

```python
orders = squad.retreat(
    threat_hex, grid, elevations,
    distance=5,               # max rings to search
    countries=None,
    food_tiers=None,
    sync=True,
    elevation_weight=0.5,     # preference for high ground
)
```

All pieces face **away** from the threat on arrival. Typical follow-up:

```python
# After executing retreat orders...
regroup = squad.form_up(FormationType.LINE, grid, elevations)
```

### retreat_path (standalone)

Single-piece retreat. Returns a `TroopPath`.

```python
path = retreat_path(piece, threat_hex, grid, elevations,
                    distance=5, countries=None, elevation_weight=0.5)
```

Runs one Dijkstra flood-fill within `distance` rings. Scores candidates by:
```
score = elevation × elevation_weight − path_cost
```
Only considers hexes in the **away hemisphere** (cube dot product < 0 with
the toward-direction). Falls back to cheapest reachable hex if cornered.

---

## Synchronization

### sync_paths

Pad instruction lists so all pieces finish on the same turn.

```python
synced = sync_paths(
    pieces,                   # all pieces referenced by orders
    orders,                   # {piece_id: InstructionList}
    end_hexes=None,           # {piece_id: final_hex} for harvest check
    food_tiers=None,          # enables productive padding
    harvest_threshold=3,      # min food tier for harvest padding
    min_turns=0,              # floor (use leader's turns as floor)
)
```

**Padding priority:**
1. If `food_tiers[end_hex] >= threshold` → `HARVEST, GIVE, HARVEST, GIVE...`
2. Otherwise → `PAUSE, PAUSE, PAUSE...`

Called automatically by `form_up`, `march_to`, `surround`, and `retreat`
when `sync=True`.

---

## Helper Functions

```python
# Direction from one hex toward another (returns 0–5)
facing = _facing_toward(grid, from_idx, to_idx)

# Opposite direction (180°)
facing = _facing_away(grid, from_idx, to_idx)

# Effective sight with elevation bonus
sight = piece.effective_sight(elevations, elevation_mult=0.005)

# Formation slot offsets in cube coordinates
offsets = _formation_offsets(FormationType.WEDGE, n_followers, leader_facing)
```

---

## Recommended Combos

### Escort a Queen to a Settlement Site

```python
# Queen paths to destination
queen.instructions = queen.pathfind_to_rules(site, grid, elevations)

# Escort forms up around queen, then marches together
orders = escort_squad.march_to(
    site, FormationType.BOX, grid, elevations,
    leader=queen, food_tiers=food_tiers,
)
for p in escort_squad.alive:
    if p.id in orders:
        p.instructions = orders[p.id]
```

### Defend a Settlement

```python
orders = garrison.surround(
    settlement.location, grid, elevations,
    mode=SurroundMode.GUARD, max_ring=2,
)
```

### Supply Chain: Harvest + Deliver

```python
# Pawn finds food, paths there, harvests, returns to queen
food_path = pawn.pathfind_to_food(grid, elevations, food_tiers)
return_path = pawn.pathfind(queen.location, grid, elevations,
                            source=food_path.end)

rules = (food_path.to_rules(grid, pawn.facing)
         + [Instruction.HARVEST.value] * 3
         + return_path.to_rules(grid, _facing_toward(grid, food_path.end, queen.location))
         + [Instruction.GIVE.value])

pawn.instructions = InstructionList(rules, patrol=True)
```

### Retreat + Regroup

```python
# Phase 1: scatter away from threat
retreat_orders = squad.retreat(enemy_hex, grid, elevations, distance=6)
for p in squad.alive:
    if p.id in retreat_orders:
        p.instructions = retreat_orders[p.id]

# Phase 2 (after retreat executes): regroup in defensive line
regroup = squad.form_up(FormationType.LINE, grid, elevations)
```

### Pincer: Two Squads from Opposite Sides

```python
# Left wing approaches from the west
left_orders = left_squad.march_to(
    west_of_target, FormationType.LINE, grid, elevations)

# Right wing approaches from the east
right_orders = right_squad.march_to(
    east_of_target, FormationType.LINE, grid, elevations)

# Sync both wings to arrive simultaneously
all_pieces = left_squad.alive + right_squad.alive
all_orders = {**left_orders, **right_orders}
synced = sync_paths(all_pieces, all_orders)
```

### Scout Patrol (Loop)

```python
# Knight patrols a triangle: forward, turn, forward, turn...
knight.instructions = InstructionList([
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
], patrol=True)  # loops forever
```

---

## Design Notes

**Hungarian matching** (`scipy.optimize.linear_sum_assignment`) is used
everywhere pieces need optimal assignment to positions: surround, form_up,
march_to. Cost matrix is `pathfind_cost[piece, slot]`. Complexity O(n³)
but n is squad size (typically 3–8), so it's instant.

**Occupied set is deliberately omitted** from plan overlays. Two pieces
can path through the same hex in preview. This is intentional — plans are
pre-execution previews, and collision resolution happens during `simulate()`.

**`form_up` vs `march_to`:**
- `form_up` — positions followers around leader's CURRENT hex. Leader not included in orders.
- `march_to` — positions everyone around a DESTINATION hex. Leader IS included in orders.

**`sync_paths` is composable.** Any function that returns
`dict[str, InstructionList]` can be synced. You can also sync orders from
different sources by merging the dicts before calling `sync_paths`.

**`retreat` breaks formation intentionally.** Each piece finds its own
best escape hex. Regroup afterward with `form_up`. Trying to maintain
formation during retreat is expensive and usually tactically wrong.

**Elevation matters in two places:**
1. Movement cost: uphill = 2×
2. Sight range: `effective_sight = base_sight + elevation × 0.005`

**Food integration:** All sync functions accept `food_tiers`. When a
piece's end hex has good food, slack turns become HARVEST+GIVE cycles
instead of idle PAUSEs. This means early-arriving units productively
farm while waiting for slower friends.
```

This covers the full API surface, the key design decisions, and the most useful composition patterns. Drop it in your `docs/` folder or as a `MANEUVERS.md` at the project root.

In [ ]:
#| default_exp game/tactics

In [ ]:
#| export
from HexMagic.database import ZoomResult, GeoStorageDebugger,GeoStorage, SaveResult, LoadResult, ChunkCover, User
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.core import Terrain, DrainageBasins
from HexMagic.styles import StyleCSS, SVGBuilder
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex, HexWrapper
from HexMagic.styles import StyleCSS,  SVGBuilder, SVGDef
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord ,HexDragMap, HexTouchMap, HexRegion
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import TerrainDisplay, CreamOverlay, RiverOverlay, ClimateOverlay, OverlayContext
from HexMagic.water.soil import SoilSystem

In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc
import fasthtml.components as fc
import httpx
import random
import pandas as pd
import threading
from dataclasses import dataclass
from datetime import datetime
import numpy as np
import math
from dataclasses import dataclass, field
#| export
import heapq

In [ ]:
#| export

from HexMagic.game.globals import appRoutes,  webMe, globalStore, ensure_user,new_game_page, create_game, create_world, invalidate_cache, showUsers, logging
from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame, CountryFlag, _map_point, Squad
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList
from HexMagic.game.data import SettlementOverlay, CountryBordersOverlay, KingdomNamesOverlay
from HexMagic.game.piece import CountryFlag, DiagramGlyphs,PieceBoard, PieceStep, _move_cost,PieceBoardPlan, PieceList
from scipy.optimize import linear_sum_assignment
from enum import Enum

In [ ]:
from HexMagic.game.food import FoodYield, FoodOverlay,FoodProfile, TroopPath, SquadVisionOverlay, SurroundMode
from HexMagic.game.piece import GroupedPieceList, SquadPlanOverlay, piece_plan_overlay, SquadSymbolOverlay
from HexMagic.game.piece import FormationType

In [ ]:
from HexMagic.game.food import SupplyPlanner, ColonyOptimizer, SupplyOverlay,  BattlePlan, BattlePlanOverlay, BattlePieceOverlay, SimEventType, SimEvent, FoodSimulator

In [ ]:
from HexMagic.core import Terrain

In [ ]:
myTerr = TerraDemo().japan_korea_map()#.shrinkWeather()
myGrid = myTerr.hexGrid
myBuilder = myGrid.builder
def clearTerr():
    myBuilder.layers = []
myGrid.adjustRadius(10)
rivers = myTerr.carve_to_ocean(num_lakes=0)

In [ ]:
myTerr.compute_climate()

In [ ]:
basins = DrainageBasins(myTerr)

In [ ]:
fy = FoodYield(myTerr, basins)
fy.compute()
#show(fy)

In [ ]:
#TerrainDisplay( CreamOverlay(), FoodOverlay(color="#8D6E63", n_tiers=6), RiverOverlay(), terrain=myTerr, basins=basins, )

In [ ]:
myBoard = GameBoard(myTerr,2, palette_name = "Spectral")

In [ ]:
??GameBoard.settlementOverlay

clearTerr()
myTerr.colorMap()
myGrid.update()
myBuilder.adjust("capitals",myBoard.settlementOverlay(scale=2.25))
myBuilder.adjust("names",myBoard.names_overlay())
myBuilder.show()

In [ ]:
for country in myBoard.kingdoms:
    print(country.countryName)
    print(country.settlements[0].name)

In [ ]:
myCountry = myBoard.kingdoms[0]
myHome = myCountry.settlements[0]

In [ ]:

# A few countries with different pattern indices
flags = [x.flag for x in myBoard.kingdoms]

tiers = [
    ('board', 0.8,  False, 'solid + stripe'),
    ('board', 1.5,  True,  'board wavy'),
    ('list',  1.5,  False, 'simple pattern'),
    ('list',  2.5,  True,  'list wavy'),
    ('large', 3.0,  False, 'full pattern'),
    ('large', 4.0,  True,  'large wavy'),
]

cols = len(tiers)
rows = len(flags)
col_w = 160
row_h = 130

canvas = SVGBuilder()
canvas.add_font('Cinzel')
canvas.width = cols * col_w + 40
canvas.height = rows * row_h + 60

# Column headers
for j, (sz, sc, wavy, desc) in enumerate(tiers):
    cx = 40 + j * col_w + col_w // 2
    canvas.adjust(f"hdr_{j}",
        f'<text x="{cx}" y="20" text-anchor="middle" font-size="11" '
        f'font-family="\'Cinzel\', sans-serif" fill="#333">{desc}</text>'
        f'<text x="{cx}" y="34" text-anchor="middle" font-size="10" '
        f'font-family="sans-serif" fill="#888">{sz} ×{sc}</text>')

for i, flag in enumerate(flags):
    # Row label
    ry = 55 + i * row_h + row_h // 2
    lbl = flag.labelStyle(f"rlbl_{i}")
    canvas.add_style(lbl)
    canvas.adjust(f"rlbl_{i}",
        f'<text x="5" y="{ry}" font-size="11" font-family="\'Cinzel\', sans-serif" '
        f'class="{lbl.name}">{flag.name}</text>')

    for j, (sz, sc, wavy, desc) in enumerate(tiers):
        cx = 40 + j * col_w + col_w // 2
        cy = 55 + i * row_h + row_h // 2

        fid = f"f_{i}_{j}"

        # Backdrop circle
        r = sc * 16
        canvas.adjust(f"bg_{i}_{j}",
            f'<circle cx="{cx}" cy="{cy}" r="{r + 6}" fill="white" '
            f'stroke="{flag.comp}" stroke-width="0.8" opacity="0.4"/>')

        flag.draw_flag(
            MapCord(cx, cy), canvas,
            scale=sc, size=sz, flag_id=fid, layer=f"flag_{i}_{j}",
            wavy=wavy, pole=(sc >= 1.5),
        )

canvas.show()

## Squads and pieces

In [ ]:
mySquads = Squad.squads(6,myCountry.flag)

In [ ]:
for squad in mySquads:
    squad.countryName = myCountry.countryName
queenGuard = mySquads[0]

### Pieces

In [ ]:
for pType in [PieceType.PAWN,PieceType.PAWN,PieceType.PAWN,PieceType.PAWN,PieceType.PAWN,PieceType.QUEEN]:
    piece = myHome.recruit(pType,flag=queenGuard.flag)
    FoodProfile.apply(piece)
    queenGuard.pieces.append(piece)

In [ ]:
show(queenGuard)

In [ ]:
#| export
@patch
def place_squad_spiral(settlement:Settlement,squad,  terrain, max_ring=10):
    """Place squad pieces in spiral around settlement, skipping water and occupied hexes.
    
    Args:
        squad: Squad with pieces to place
        settlement: Settlement to spiral around  
        terrain: Terrain object (has hexGrid, elevations)
        max_ring: Maximum spiral radius to search
        
    Returns:
        Number of pieces successfully placed
    """
    grid = terrain.hexGrid
    center = settlement.location
    
    # Collect occupied hexes: settlement location + all citizen locations
    occupied = {center}
    for citizen in settlement.citizens:
        if citizen.location is not None:
            occupied.add(citizen.location)
    
    # Generate spiral outward from center
    spiral = HexPosition.origin().spiral(max_ring)
    
    placed = 0
    for hp in spiral[1:]:  # skip center (settlement itself)
        if placed >= len(squad.pieces):
            break
        
        idx = grid.hexposition_to_index(hp, origin_index=center)
        if idx < 0:
            continue
        
        # Skip water (elevation <= 0)
        if terrain.elevations[idx] <= 0:
            continue
        
        # Skip occupied by settlement citizens
        if idx in occupied:
            continue
        
        squad.pieces[placed].location = idx
        occupied.add(idx)  # prevent stacking within the squad too
        placed += 1
    
    return placed


In [ ]:
n = myHome.place_squad_spiral(queenGuard,  myTerr, max_ring=6)
print(f"Placed {n}/{len(queenGuard.pieces)} pieces")
for person in queenGuard.pieces:
    print(f"  {person.name} at {person.location}")


In [ ]:
GroupedPieceList(queenGuard.pieces,grid=myGrid)

In [ ]:
#svg = food_overlay(fy)
myTerr.hexGrid.builder.layers = []
myTerr.terrainCream()
#myTerr.hexGrid.builder.adjust("food", svg)
show(myTerr.hexGrid.builder)


In [ ]:
clearTerr()

In [ ]:
def PieceOverlay(squads, **kw) -> OverlaySpec:
    """Draw each living piece at its current location."""
    def render(ctx) -> str:
        grid = ctx.grid
        c2f = getattr(ctx, 'c2f', None)
        N = len(grid.hexes)
        parts = []
        for squad in squads:
            for piece in (squad.alive if hasattr(squad, 'alive') else squad.pieces):
                if piece.location is None: continue
                idx = _map_point(piece.location, c2f)
                if 0 <= idx < N:
                    parts.append(piece.draw_svg(grid, hex_idx=idx))
        return '\n'.join(parts)
    return OverlaySpec("pieces", render, priority=80)


In [ ]:
show(myBuilder)

Does the terrainZoom work? should it create a mapping?

Can you give me the proper versions of zoomed and TerrainDisplay?

from HexMagic.overlay import OverlayContext

@patch
def zoom(self: Terrain, region: HexRegion, padding: int = 1) -> 'Terrain':
    """Create a new Terrain zoomed into a HexRegion, with c2f mapping."""
    new_grid, _, mapper = region.crop_to_centered_grid(
        style=self.hexGrid.style, padding=padding
    )
    if new_grid is None:
        zoomed = self.clone()
        zoomed.c2f = None
        return zoomed

    zoomed = self.clone()
    zoomed.assign_grid(new_grid)

    # Build reverse mapping: old_idx → [new_idx] (list to match _map_point API)
    c2f = {}
    for new_idx in range(len(new_grid.hexes)):
        old_idx = mapper(new_idx)
        if 0 <= old_idx < len(self.elevations):
            zoomed.elevations[new_idx] = self.elevations[old_idx]
            for k in self.fields:
                zoomed.fields[k][new_idx] = self.fields[k][old_idx]
            c2f[old_idx] = [new_idx]

    zoomed.c2f = c2f
    return zoomed


def TerrainDisplay(
    *overlays: OverlaySpec,
    terrain: 'Terrain',
    result=None, board=None, c2f=None,
    basins=None, soil=None,
    radius=None, debug=False,
    region: 'HexRegion' = None,
    padding: int = 1,
    **attrs
):
    # Zoom if region provided
    if region is not None:
        terrain = terrain.zoom(region, padding=padding)
        # Pick up the c2f from zoom unless caller provided one explicitly
        if c2f is None:
            c2f = getattr(terrain, 'c2f', None)

    grid = terrain.hexGrid
    if radius: grid.adjustRadius(radius)

    if basins is None and result and hasattr(result, 'basins'):
        basins = result.basins
    if soil is None and result and hasattr(result, 'soil'):
        soil = result.soil

    ctx = OverlayContext(
        terrain=terrain, grid=grid, builder=grid.builder,
        extras=dict(result=result, board=board, c2f=c2f,
                    basins=basins, soil=soil)
    )

    available = {'terrain', 'grid', 'builder'}
    if soil:   available.add('soil')
    if result: available.add('result')
    if board:  available.add('board')
    if c2f:    available.add('c2f')
    if basins: available.add('basins')

    # Phase 1: Run renderers, collect SVG strings (+ hex style side effects)
    collected = []
    sorted_overlays = sorted(overlays, key=lambda o: o.priority)
    for spec in sorted_overlays:
        missing = spec.requires - available
        if missing:
            logging.warning(f"Skipping {spec.name}: missing {missing}")
            continue
        try:
            svg = spec.renderer(ctx)
            if svg: collected.append((spec.name, svg))
        except Exception as e:
            logging.error(f"Overlay {spec.name} failed: {e}")

    # Phase 2: Bake hex styles — stylized (organic regions) or flat (individual hexes)
    stylized_spec = next((s for s in sorted_overlays if s.stylized), None)
    if stylized_spec:
        f = stylized_spec.edge_fn or unique_windy_edge()
        hexes_svg = grid.styleLayer(f=f, smooth=True)
        grid.builder.adjust("hexes", hexes_svg)
    else:
        grid.update()

    # Phase 3: Add SVG overlay layers ON TOP of hexes
    for name, svg in collected:
        grid.builder.adjust(name, svg)

    if debug: return grid.builder.__ft__()

    return Div(
        HexTouchMap(grid, cls=attrs.pop('map_cls', 'w-full h-full')),
        **attrs
    )


In [ ]:
@patch
def settlementOverlay(self: GameBoard, terrain: Terrain = None,
                      c2f: dict = None, scale: float = 2.0) -> str:
    """Settlement markers using flags, with detail tier chosen by scale."""
    terrain = terrain or self.terrain
    grid = terrain.hexGrid
    num_hexes = len(grid.hexes)

    # Pick detail tier based on scale
    if scale < 1.5:
        size = 'board'
    elif scale < 3.0:
        size = 'list'
    else:
        size = 'large'

    overlay = ""
    for country in self.kingdoms:
        if country.flag is None:
            continue

        for j, s in enumerate(country.settlements):
            if s.location is None:
                continue
            local_idx = _map_point(s.location, c2f)
            if local_idx < 0 or local_idx >= num_hexes:
                continue

            coords = grid.hexes[local_idx].center
            piece_id = f"flag_{country.countryId}_{j}"

            svg_str, pat_def = country.flag.flag_svg(
                MapCord(coords.x, coords.y),
                scale=scale//2,
                size=size,
                flag_id=piece_id,
            )

            if pat_def is not None:
                grid.builder.add_definition(pat_def)

            overlay += svg_str + "\n"

    return overlay


In [ ]:
??SettlementOverlay

In [ ]:
clearTerr()
myGrid.adjustRadius(50)

region = HexRegion(hexes=set(myTerr.ring(myHome.location, 5)), hexGrid=myTerr.hexGrid)


TerrainDisplay(
    CreamOverlay(stylized=True),
    SquadSymbolOverlay([queenGuard]),
    SettlementOverlay(scale=5),
    KingdomNamesOverlay(),
    PieceOverlay([queenGuard]),
    terrain=myTerr,
    region=region, padding=2,
    board=myBoard
)


why no pieces on the map?

In [ ]:
zoomed = myTerr.zoom(region, padding=2)
ctx = OverlayContext(
    terrain=zoomed, grid=zoomed.hexGrid, builder=zoomed.hexGrid.builder,
    extras=dict(c2f=zoomed.c2f)
)

# Test the overlay directly (no try/except hiding errors)
overlay = PieceOverlay([queenGuard])
svg = overlay.renderer(ctx)
print(repr(svg[:200]) if svg else "EMPTY")


In [ ]:
??_map_point

Is zoomed creating the right c2f?

In [ ]:
myQueen = queenGuard.by_rank()[0]
show(myQueen)

In [ ]:
queenPos = myGrid.index_to_hexposition(myQueen.location)
destPost = queenPos + (6 * HexPosition.W)
dest = myGrid.hexposition_to_index(destPost)
dest

In [ ]:


myQueen.instructions = myQueen.pathfind_to_rules(target=dest, grid=myGrid, elevations=myTerr.elevations)
myQueen.instructions.rules.append(Instruction.PAUSE)

#myQueen.rules = queenPath.to_rules(grid=myGrid)

show(myQueen.instructions)

In [ ]:
myQueen.instructions.rules

In [ ]:
steps = myQueen.simulate(myGrid, myTerr.elevations, num_turns=12)
steps

In [ ]:
clearTerr()
myGrid.adjustRadius(25)
TerrainDisplay(
    TerrainOverlay(stylized=True),
    SquadSymbolOverlay([queenGuard]),
    SquadPlanOverlay([queenGuard]),
    terrain=myTerr,
    region=region, padding=2,
    board=myBoard
)


In [ ]:
from HexMagic.game.piece import FormationType

orders = queenGuard.march_to(
    dest, FormationType.BOX, myGrid, myTerr.elevations,
    leader=myQueen, food_tiers=fy.tiers,
)
for p in queenGuard.alive:
    if p.id in orders:
        p.instructions = orders[p.id]


In [ ]:
#!cat ../../HexMagic/game/piece.py

We have build up so primitives to do 
```python
orders = queenGuard.surround(target=dest, grid=myGrid, elevations=myTerr.elevations,mode=SurroundMode.FEED)
print(f"Assigned {len(orders)}/{len(queenGuard)} pieces to surround positions")
for p in queenGuard.alive:
    if p.id in orders:
        p.instructions = orders[p.id]

myQueen.instructions = myQueen.pathfind_to_rules(target=dest, grid=myGrid, elevations=myTerr.elevations)
myQueen.instructions.rules.append(Instruction.PAUSE)
```
what is a better way of escorting the queen? She looks after us all.

We might need to import stuff from 04_piece.ipynb

In [ ]:
clearTerr()
myGrid.adjustRadius(25)
TerrainDisplay(
    CreamOverlay(),
    SquadSymbolOverlay([queenGuard]),
    SquadVisionOverlay([queenGuard]),
    SquadPlanOverlay([queenGuard]),
    region=region, padding=2,
    terrain=myTerr,
    board=myBoard
)

In [ ]:
??SquadVisionOverlay

In [ ]:
show(myBuilder)

In [ ]:
# Zoomed
ctx = OverlayContext.from_terrain(myTerr, region=region, padding=2)
ctx.test(SquadSymbolOverlay([queenGuard]))

In [ ]:
??SquadSymbolOverlay

Why isn't anything be added into the render?

oops

In [ ]:
??SquadPlanOverlay

## Let them eat cake

In [ ]:
elevs = myTerr.elevations


# Simple food tiers: low ground = fertile, high = barren
food_tiers = np.zeros(len(elevs), dtype=int)
for i, e in enumerate(elevs):
    if   e <= 0:    food_tiers[i] = 0   # ocean
    elif e < 60:    food_tiers[i] = 7   # coastal plains  ← pawns go here
    elif e < 120:   food_tiers[i] = 5
    elif e < 180:   food_tiers[i] = 3
    elif e < 260:   food_tiers[i] = 1
    else:           food_tiers[i] = 0   # peaks


print("=== BEFORE ===")
sim = FoodSimulator(
    grid=myGrid,
    elevations=elevs,
    food_tiers=food_tiers,
    pieces=queenGuard.pieces,
)
sim.summary()

In [ ]:
show(sim)

In [ ]:
sim.run(num_turns=25)
show(sim)

In [ ]:
myPawns = queenGuard.by_rank()[1:]

In [ ]:
for i in range(16):
    for piece in queenGuard:
        piece.instructions.rules.append(Instruction.HARVEST)
        piece.instructions.rules.append(Instruction.HARVEST)
        piece.instructions.rules.append(Instruction.GIVE)

In [ ]:
#| export
sim.run(num_turns=25)
show(sim)

In [ ]:
show(myQueen.instructions)

In [ ]:
show(myQueen)

## Another Squad

In [ ]:
roadsters = mySquads[1]
for pType in [PieceType.PAWN,PieceType.PAWN,PieceType.PAWN,PieceType.PAWN,PieceType.PAWN,PieceType.BISHOP,PieceType.BISHOP,PieceType.BISHOP,PieceType.BISHOP,PieceType.BISHOP]:
    piece = myHome.recruit(pType,flag=queenGuard.flag)
    FoodProfile.apply(piece)
    roadsters.pieces.append(piece)

In [ ]:
show(roadsters)

In [ ]:
n = myHome.place_squad_spiral(roadsters, myTerr, max_ring=6)
print(f"Placed {n}/{len(roadsters.pieces)} pieces")

In [ ]:
GroupedPieceList(roadsters,grid=myGrid)

In [ ]:
PieceList??

In [ ]:
GroupedPieceList??

why no distances shown?

In [ ]:
clearTerr()
TerrainDisplay(
    CreamOverlay(),
    SquadSymbolOverlay([roadsters,queenGuard]),
    PieceOverlay([roadsters,queenGuard]),
    SettlementOverlay(scale=5),
    region= region,
    terrain=myTerr,
    board = myBoard
    
)


In [ ]:
ctx = OverlayContext.from_terrain(myTerr, region=region, padding=2)
ctx.test(SquadSymbolOverlay([roadsters,queenGuard]))

In [ ]:
!cat ../../HexMagic/overlay.py

What is going on with the Squad Symbol overlay?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Looking at the evidence, the `SquadSymbolOverlay` **is generating valid SVG** — your `ctx.test()` confirms it produces 10,247 chars of SVG with two squad icons ("Pyro Pigs" at ~328,250 and "Thunder Toucans" at ~90,212) with animal silhouettes and labels.

**But the `TerrainDisplay` output is missing the layer entirely.** The final SVG only has `<g data-layer="root">` and `<g data-layer="hexes">` — no squad symbols layer.

The pipeline in `TerrainDisplay` is:

1. ✅ Renderer runs → collects SVG into `collected` list
2. ✅ `grid.update()` bakes hex styles
3. ⚠️ `grid.builder.adjust(name, svg)` — adds the layer to the builder
4. ❓ `HexTouchMap(grid, ...)` — renders the final SVG

The likely culprit is step 4: **`HexTouchMap` may regenerate the SVG from the grid without including the builder's adjusted layers**, or it captures the builder state *before* the `adjust` calls take effect.

You could verify by checking whether `HexTouchMap` uses `grid.builder` to render, or if it builds its own SVG independently. A quick diagnostic:

```python
# After TerrainDisplay runs, check if the builder actually has the layer:
print([l for l in myGrid.builder.layers])  # or however layers are stored
```

Alternatively, try passing `debug=True` to `TerrainDisplay` — that returns `grid.builder.__ft__()` directly instead of going through `HexTouchMap`, which should include the adjusted layers.

In [ ]:
# After TerrainDisplay runs, check if the builder actually has the layer:
print([l for l in myGrid.builder.layers])  # or however layers are stored


In [ ]:
!tail -10 base.text

It really doesn't want to work

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The problem is that `TerrainDisplay` **silently swallows exceptions**:

```python
except Exception as e:
    logging.error(f"Overlay {spec.name} failed: {e}")
```

Meanwhile `ctx.test()` propagates them. So the overlay probably *is* erroring in the `TerrainDisplay` context but you can't see it.

Try enabling logging first:

```python
import logging
logging.basicConfig(level=logging.WARNING, force=True)
```

Then re-run the `TerrainDisplay` call. Or even better, reproduce the **exact** context `TerrainDisplay` creates internally and use `.test()` which won't swallow the error:

```python
# Mimic what TerrainDisplay does internally
terrain_z = myTerr.zoom(region, padding=1)  # Note: TD uses padding=1 by default, you used 2 in ctx.test
grid_z = terrain_z.hexGrid
c2f_z = getattr(terrain_z, 'c2f', None)

ctx2 = OverlayContext(
    terrain=terrain_z, grid=grid_z, builder=grid_z.builder,
    extras=dict(result=None, board=None, c2f=c2f_z, basins=None, soil=None)
)
ctx2.test(SquadSymbolOverlay([roadsters, queenGuard]))
```

Notice the `padding=1` vs your earlier working test that used `padding=2` — `TerrainDisplay` defaults to `padding=1`, so the zoomed grid is slightly different. That mismatch could cause pieces to fall outside the grid bounds and trigger an error that gets eaten.

In [ ]:

# ── Create planner ─────────────────────────────────────────────────────────
planner = SupplyPlanner(myTerr, settlement=myHome.location)
planner.summary(n_pawns=6, n_bishops=3)

# ── Show throughput curves + bishops-needed heatmap ────────────────────────
show(planner)

# ── Find some expansion targets at various distances ───────────────────────
targets = []
for min_cost in [6, 12, 20, 30]:
    t = planner.find_target(min_elev=100, max_tier=2, min_cost=min_cost)
    if t and t not in [x[0] for x in targets]:
        route = planner.plan_route(t)
        targets.append((t, route))
        print(f"Target @ hex {t}: cost={route['cost']:.1f}  "
              f"travel={route['travel_turns']} turns  "
              f"bishops_needed={route['bishops_needed']}  "
              f"net_delivery={route['net_delivery']:.1f}")

# ── Create fake consumer pieces at those targets ──────────────────────────
consumers = []
piece_types = [PieceType.QUEEN, PieceType.KNIGHT, PieceType.ROOK, PieceType.KNIGHT]
for i, (hex_idx, route) in enumerate(targets):
    pt = piece_types[i % len(piece_types)]
    p = Piece(location=hex_idx, piece_type=pt, name=f"Target_{i}")
    FoodProfile.apply(p)
    consumers.append(p)
    print(f"  {p.name} ({pt.name}): diet={p.diet} @ hex {hex_idx}")

# ── Optimize colony allocation ─────────────────────────────────────────────
opt = ColonyOptimizer(planner, max_pawns=6, max_bishops=5)
analyzed = opt.analyze_consumers(consumers)
result = opt.optimize(analyzed)
opt._last_result = result
opt.print_plan(result)
show(opt)


In [ ]:
planner = SupplyPlanner(myTerr, settlement=myHome.location)
t = planner.find_target(min_elev=80, max_tier=3, min_cost=min_cost)
t

### need better targets

In [ ]:
#this will be added later
@patch
def supply_line(feeders: Squad, path: list[int], 
                grid: HexGrid, elevations: np.ndarray,
                food_tiers: np.ndarray,
                min_tier: int = 2,
                max_offset: int = 2,
                countries: np.ndarray = None,
                elevation_mult: float = 0.005
                ) -> dict[str, InstructionList]:
    """Station feeders on fertile tiles along a march path.
    
    Args:
        path: Hex indices of the march route
        feeders: Squad of available pieces to assign as suppliers
        grid: HexGrid
        elevations: Elevation array
        food_tiers: Per-hex food tier array
        min_tier: Minimum food tier for a candidate station
        max_offset: Max ring distance from path for stations
        countries: Country array (optional, for impassable)
        elevation_mult: Sight bonus per elevation unit
    
    Returns:
        Dict mapping piece.id → InstructionList (path + rotate + harvest/give loop)
    """
    pieces = feeders.alive
    if not pieces:
        return {}
    
    path_set = set(path)
    
    # --- Phase 1: Generate scored candidate stations ---
    # candidates[hex_idx] = (facing, tier, covered_path_hexes)
    candidates = {}
    
    for path_hex in path:
        for ring in range(1, max_offset + 1):
            for hp in HexPosition.origin().ring(ring):
                idx = grid.hexposition_to_index(hp, path_hex)
                if idx < 0 or idx in grid.invalidRegion:
                    continue
                if elevations[idx] <= 0:
                    continue
                if idx in path_set:
                    continue
                if int(food_tiers[idx]) < min_tier:
                    continue
                if countries is not None and countries[idx] < 0:
                    continue
                
                facing = _facing_toward(grid, idx, path_hex)
                facing_dir = HexPosition.directions()[facing]
                
                elev = max(0, elevations[idx])
                eff_sight = 3 + elev * elevation_mult
                fov = set(field_of_view(HexPosition.origin(), facing_dir, int(eff_sight)))
                
                covered = {ph for ph in path
                           if grid.index_to_hexposition(ph, idx) in fov}
                
                if not covered:
                    continue
                
                tier = int(food_tiers[idx])
                if idx not in candidates or len(covered) > len(candidates[idx][2]):
                    candidates[idx] = (facing, tier, covered)
    
    if not candidates:
        return {}
    
    # --- Phase 2: Greedy set cover → pick station positions ---
    uncovered = set(path)
    selected = []      # [(hex_idx, facing, tier), ...]
    remaining = dict(candidates)
    
    while uncovered and remaining:
        best_idx = max(remaining, key=lambda i: (
            len(remaining[i][2] & uncovered) * remaining[i][1]
        ))
        facing, tier, covered = remaining.pop(best_idx)
        newly_covered = covered & uncovered
        if not newly_covered:
            continue
        selected.append((best_idx, facing, tier))
        uncovered -= newly_covered
    
    if not selected:
        return {}
    
    # --- Phase 3: Hungarian matching (single pathfind per pair) ---
    INF = 1e9
    n_pieces = len(pieces)
    n_stations = len(selected)
    cost = np.full((n_pieces, n_stations), INF)
    paths: dict[tuple[int, int], TroopPath] = {}
    
    for i, piece in enumerate(pieces):
        if piece.location is None or piece.location < 0:
            continue
        for j, (pos_idx, _, _) in enumerate(selected):
            tp = piece.pathfind(pos_idx, grid, elevations, countries)
            if not tp:
                continue
            paths[(i, j)] = tp
            cost[i, j] = tp.cost
    
    row_ind, col_ind = linear_sum_assignment(cost)
    
    # --- Phase 4: Build instructions from cached paths ---
    assignments = {}
    for i, j in zip(row_ind, col_ind):
        if cost[i, j] >= INF:
            continue
        
        piece = pieces[i]
        tp = paths[(i, j)]
        _, facing, _ = selected[j]
        
        rules = tp.to_rules(grid, piece.facing)
        
        # Rotate to face the path
        end_facing = _facing_toward(grid, tp[-2], tp[-1])
        diff = (facing - end_facing) % 6
        if diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))
        
        # Station action loop
        rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
        assignments[piece.id] = InstructionList(rules, cursor=0, patrol=True)
    
    return assignments

In [ ]:
def SupplyOverlay(settlement, field='bishops_needed', planner=None,
                  cmap_name='RdYlGn_r', opacity=0.35, **kw) -> OverlaySpec:
    """Heatmap of supply reachability from a settlement.

    field: 'bishops_needed' | 'net_delivery' | 'path_cost' | 'travel_turns'
    """

    def render(ctx):
        nonlocal planner
        if planner is None:
            planner = SupplyPlanner(ctx.terrain, settlement=settlement)

        srm = planner.supply_range_map()
        if not srm:
            return ""

        grid = ctx.grid

        # Compute field values
        field_vals = {}
        for idx, (cost, bn) in srm.items():
            if field == 'bishops_needed':
                field_vals[idx] = min(bn, 8)
            elif field == 'net_delivery':
                field_vals[idx] = planner.net_delivery(cost)
            elif field == 'path_cost':
                field_vals[idx] = cost
            elif field == 'travel_turns':
                field_vals[idx] = planner.travel_turns(cost)

        if not field_vals:
            return ""

        vals = list(field_vals.values())
        vmin, vmax = min(vals), max(vals)
        if field == 'net_delivery':
            cmap = plt.cm.get_cmap('RdYlGn')     # green=high delivery=good
        else:
            cmap = plt.cm.get_cmap(cmap_name)     # red=high cost=bad

        svg = ""
        for idx, val in field_vals.items():
            if idx < 0 or idx >= len(grid.hexes):
                continue
            t = (val - vmin) / (vmax - vmin) if vmax > vmin else 0.5
            rgba = cmap(t)
            color = f"rgb({int(rgba[0]*255)},{int(rgba[1]*255)},{int(rgba[2]*255)})"
            h = grid.hexes[idx]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in h.vertices())
            svg += (f'<polygon points="{pts}" fill="{color}" '
                    f'opacity="{opacity:.2f}" stroke="none"/>\n')

        # Settlement marker
        sc = grid.hexes[settlement].center
        r = grid.radius if hasattr(grid, 'radius') else 20
        svg += (f'<circle cx="{sc.x:.0f}" cy="{sc.y:.0f}" r="{r*0.6:.0f}" '
                f'fill="#FFD700" stroke="#333" stroke-width="2" opacity="0.9"/>\n')
        svg += (f'<text x="{sc.x:.0f}" y="{sc.y+4:.0f}" text-anchor="middle" '
                f'font-size="{r*0.6:.0f}" font-weight="bold" fill="#333">⛺</text>\n')

        return svg

    return OverlaySpec("supply_range", render, priority=42)


def ColonyPlanOverlay(planner, result, **kw) -> OverlaySpec:
    """Show colony optimization result: settlement, routes, and target markers.

    Args:
        planner: SupplyPlanner (provides settlement + pathfinding)
        result:  dict from ColonyOptimizer.optimize()
    """

    def render(ctx):
        grid = ctx.grid
        r = grid.radius if hasattr(grid, 'radius') else 20
        svg = ""

        settlement = planner.settlement
        sc = grid.hexes[settlement].center

        assignments = result.get('assignments', [])
        if not assignments:
            return ""

        for a in assignments:
            consumer = a['consumer']
            target_idx = consumer.hex_idx
            if target_idx < 0 or target_idx >= len(grid.hexes):
                continue

            tc = grid.hexes[target_idx].center
            fed = a['fed']
            n_bish = a['bishops']

            # ── Route line ──
            dummy = Piece(location=settlement)
            path = dummy.pathfind(target_idx, grid, planner.elevations)
            if path and len(path) > 1:
                route_color = "#2ecc71" if fed else "#e74c3c"
                points = []
                for hex_idx in path.hexes:
                    if 0 <= hex_idx < len(grid.hexes):
                        hc = grid.hexes[hex_idx].center
                        points.append(f"{hc.x:.0f},{hc.y:.0f}")
                if points:
                    svg += (f'<polyline points="{" ".join(points)}" '
                            f'fill="none" stroke="{route_color}" '
                            f'stroke-width="{max(2, r*0.12):.1f}" '
                            f'stroke-dasharray="6,4" '
                            f'stroke-opacity="0.7" '
                            f'marker-end="url(#arrowhead_{route_color[1:]})"/>\n')

            # ── Target marker ──
            marker_color = "#2ecc71" if fed else "#e74c3c"
            marker_r = r * 0.55
            svg += (f'<circle cx="{tc.x:.0f}" cy="{tc.y:.0f}" r="{marker_r:.0f}" '
                    f'fill="{marker_color}" fill-opacity="0.25" '
                    f'stroke="{marker_color}" stroke-width="2.5"/>\n')

            # ── Piece type icon ──
            ptype = getattr(consumer.piece, 'piece_type', None)
            icon = ptype.icon if ptype else "?"
            svg += (f'<text x="{tc.x:.0f}" y="{tc.y+r*0.15:.0f}" '
                    f'text-anchor="middle" font-size="{r*0.55:.0f}" '
                    f'fill="{marker_color}" font-weight="bold">{icon}</text>\n')

            # ── Bishop count label ──
            if n_bish > 0:
                lx = tc.x + marker_r * 0.9
                ly = tc.y - marker_r * 0.7
                svg += (f'<circle cx="{lx:.0f}" cy="{ly:.0f}" r="{r*0.28:.0f}" '
                        f'fill="#3498db" stroke="#fff" stroke-width="1.5"/>\n')
                svg += (f'<text x="{lx:.0f}" y="{ly+r*0.1:.0f}" '
                        f'text-anchor="middle" font-size="{r*0.3:.0f}" '
                        f'fill="white" font-weight="bold">{n_bish}</text>\n')

        # ── Arrowhead defs ──
        for color_hex in ['2ecc71', 'e74c3c']:
            aid = f"arrowhead_{color_hex}"
            arrow_def = SVGDef("marker", aid,
                f'<path d="M0,0 L0,6 L6,3 z" fill="#{color_hex}"/>',
                markerWidth=6, markerHeight=6,
                refX=5, refY=3,
                orient="auto", markerUnits="strokeWidth")
            ctx.builder.add_definition(arrow_def)

        # ── Settlement marker (on top) ──
        svg += (f'<circle cx="{sc.x:.0f}" cy="{sc.y:.0f}" r="{r*0.65:.0f}" '
                f'fill="#FFD700" stroke="#333" stroke-width="2.5" opacity="0.9"/>\n')
        svg += (f'<text x="{sc.x:.0f}" y="{sc.y+r*0.18:.0f}" text-anchor="middle" '
                f'font-size="{r*0.6:.0f}" font-weight="bold" fill="#333">⛺</text>\n')

        return svg

    return OverlaySpec("colony_plan", render, priority=72)


In [ ]:
queenPos = myGrid.index_to_hexposition(dest)
destPost = queenPos + (9 * HexPosition.SW)
higher = myGrid.hexposition_to_index(destPost)


In [ ]:
orders = queenGuard.surround(target=higher, grid=myGrid, elevations=myTerr.elevations,mode=SurroundMode.FEED)
print(f"Assigned {len(orders)}/{len(queenGuard)} pieces to surround positions")
for p in queenGuard.alive:
    if p.id in orders:
        p.instructions = orders[p.id]

myQueen.instructions = myQueen.pathfind_to_rules(target=higher, grid=myGrid, elevations=myTerr.elevations)
myQueen.instructions.rules.append(Instruction.PAUSE)

In [ ]:
??TroopPath

can we use the roadsters to create a path for the queen that countinues going west using supply_line

In [ ]:
clearTerr()
TerrainDisplay(
    CreamOverlay(),
    SupplyOverlay(settlement=myHome.location, planner=planner),
    SquadSymbolOverlay([roadsters,queenGuard]),
    PieceOverlay([roadsters,queenGuard]),
    terrain=myTerr
)
show(myBuilder)

Why no supply overlay?

In [ ]:
class SurroundMode(Enum):
    FEED  = "feed"   # face inward — supply chain
    GUARD = "guard"  # face outward — defensive perimeter


@patch
def effective_sight(self: Piece, elevations: np.ndarray,
                    elevation_mult: float = 0.005) -> float:
    """Sight range including elevation bonus."""
    if self.location is None or self.location < 0:
        return float(self.sight)
    elev = max(0, elevations[self.location])
    return self.sight + elev * elevation_mult


def _facing_toward(grid: HexGrid, from_idx: int, to_idx: int) -> int:
    """Direction index (0–5) from from_idx toward to_idx."""
    rel = grid.index_to_hexposition(to_idx, from_idx)
    # Find closest cardinal direction
    dirs = HexPosition.directions()
    best_dir = 0
    best_dot = -999
    for i, d in enumerate(dirs):
        # Dot product in cube coords
        dot = rel.q * d.q + rel.r * d.r + rel.s * d.s
        if dot > best_dot:
            best_dot = dot
            best_dir = i
    return best_dir


def _facing_away(grid: HexGrid, from_idx: int, to_idx: int) -> int:
    """Direction index (0–5) pointing away from to_idx."""
    return (_facing_toward(grid, from_idx, to_idx) + 3) % 6


def surround(pieces: list[Piece], target: int,
             grid: HexGrid, elevations: np.ndarray,
             mode: SurroundMode = SurroundMode.GUARD,
             max_ring: int = 3,
             countries: np.ndarray = None,
             elevation_mult: float = 0.005
             ) -> dict[str, InstructionList]:
    """Assign pieces to surround positions using Hungarian matching.
    
    Args:
        pieces: Pieces to assign (must have locations)
        target: Hex index to surround
        grid: HexGrid
        elevations: Elevation array
        mode: FEED (face inward) or GUARD (face outward)
        max_ring: Maximum ring distance from target
        countries: Country array (optional, for impassable)
        elevation_mult: Sight bonus per elevation unit
    
    Returns:
        Dict mapping piece.id → InstructionList with path + final rotation
    """
    # 1. Generate candidate positions (rings 1..max_ring)
    candidates = []
    for ring in range(1, max_ring + 1):
        for hp in HexPosition.origin().ring(ring):
            idx = grid.hexposition_to_index(hp, target)
            if idx < 0 or idx in grid.invalidRegion:
                continue
            if elevations[idx] <= 0:
                continue
            if countries is not None and countries[idx] < 0:
                continue
            candidates.append((idx, ring))
    
    if not candidates or not pieces:
        return {}
    
    n_pieces = len(pieces)
    n_positions = len(candidates)
    
    # 2. Build cost matrix: pieces × positions
    INF = 1e9
    cost = np.full((n_pieces, n_positions), INF)
    
    for i, piece in enumerate(pieces):
        if piece.location is None or piece.location < 0:
            continue
        
        # Effective sight at current location (conservative — use current elev)
        eff_sight = piece.effective_sight(elevations, elevation_mult)
        
        for j, (pos_idx, ring_dist) in enumerate(candidates):
            # Can piece see target from this position?
            if ring_dist > eff_sight:
                continue  # can't see target from here
            
            # Pathfind cost
            path = piece.pathfind(pos_idx, grid, elevations, countries)
            if not path:
                continue  # unreachable
            
            # Cost = path length (weighted by movement costs)
            path_cost = 0.0
            for k in range(len(path) - 1):
                path_cost += _move_cost(elevations, path[k], path[k + 1])
            
            cost[i, j] = path_cost
    
    # 3. Hungarian matching
    row_ind, col_ind = linear_sum_assignment(cost)
    
    # 4. Generate instruction lists
    assignments = {}
    for i, j in zip(row_ind, col_ind):
        if cost[i, j] >= INF:
            continue  # unmatched
        
        piece = pieces[i]
        pos_idx = candidates[j][0]
        
        path = piece.pathfind(pos_idx, grid, elevations, countries)
        if len(path) < 2:
            continue
        
        rules = InstructionList.path_to_rules(path, grid, start_facing=piece.facing)
        
        # Append final facing rotation
        if mode == SurroundMode.FEED:
            desired_facing = _facing_toward(grid, pos_idx, target)
        else:
            desired_facing = _facing_away(grid, pos_idx, target)
        
        # Calculate rotations needed to reach desired facing
        # Current facing after path (last rule's facing)
        # Simulate to get final facing — or just compute from path end
        path_end_facing = _facing_toward(grid, path[-2], path[-1]) if len(path) >= 2 else piece.facing
        
        # Shortest rotation to desired
        diff = (desired_facing - path_end_facing) % 6
        if diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))
        
        assignments[piece.id] = InstructionList(rules, cursor=0, patrol=False)
    
    return assignments

Can you build a demo using a gameboard that shows a bunch of peices moving to feed the queen?

In [ ]:
# --- Build the board ---
board = PieceBoard.hilly(rings=7, radius=20, seed=42)
grid = board.grid
elevs = board.elevations

# Place the queen near center on high ground
mid = grid.middle
queen = board.add_piece(mid, PieceType.QUEEN, facing=0, country_id=1,
                        instructions=InstructionList([Instruction.PAUSE.value], patrol=True))
queen.food = 2.0  # hungry queen

# Scatter pawns and a knight around — various distances
offsets = [
    (-3, -2), (-4, 1), (2, -3), (3, 2), (5, 0), (-2, 4), (1, -5), (-5, -1)
]
feeders = []
for i, (dq, dr) in enumerate(offsets):
    hp = HexPosition(dq, dr, -dq - dr)
    idx = grid.hexposition_to_index(hp, mid)
    if idx < 0 or elevs[idx] <= 0:
        continue
    ptype = PieceType.KNIGHT if i == 0 else PieceType.PAWN
    p = board.add_piece(idx, ptype, facing=random.randint(0, 5), country_id=1,
                        instructions=InstructionList([]))
    p.food = 8.0  # well-fed, ready to share
    feeders.append(p)

print(f"Queen at hex {mid}, {len(feeders)} feeders placed")

# --- Run surround with FEED mode ---
orders = surround(feeders, queen.location, grid, elevs,
                  mode=SurroundMode.FEED, max_ring=2)

print(f"Assigned {len(orders)} pieces to surround positions")
for p in feeders:
    if p.id in orders:
        p.instructions = orders[p.id]
        print(f"  {p.name} ({p.piece_type.name}): {len(p.instructions.rules)} instructions")

# --- Render ---
#board.render(show_plan=True, num_turns=8)
#show(board)


Fix?

How about with a pieces overlay and the queen not sitting in water?

In [ ]:
# --- Build the board ---
board = PieceBoard.hilly(rings=7, radius=20, seed=42)
grid = board.grid
elevs = board.elevations

# Find nearest land hex to center for queen
mid = grid.middle
queen_idx = None
for dist in range(0, 5):
    for hp in HexPosition.origin().ring(dist):
        idx = grid.hexposition_to_index(hp, mid)
        if idx >= 0 and elevs[idx] > 0:
            queen_idx = idx
            break
    if queen_idx is not None:
        break

print(f"Queen placed at hex {queen_idx} (elev={elevs[queen_idx]:.0f})")

queen = board.add_piece(queen_idx, PieceType.QUEEN, facing=0, country_id=1,
                        instructions=InstructionList([Instruction.PAUSE.value], patrol=True))
queen.food = 2.0

# Scatter feeders — skip any that land on water
offsets = [
    (-3, -2), (-4, 1), (2, -3), (3, 2), (5, 0), (-2, 4), (1, -5), (-5, -1)
]
feeders = []
for i, (dq, dr) in enumerate(offsets):
    hp = HexPosition(dq, dr, -dq - dr)
    idx = grid.hexposition_to_index(hp, queen_idx)
    if idx < 0 or elevs[idx] <= 0:
        continue
    ptype = PieceType.KNIGHT if i == 0 else PieceType.PAWN
    p = board.add_piece(idx, ptype, facing=random.randint(0, 5), country_id=1,
                        instructions=InstructionList([]))
    p.food = 8.0
    feeders.append(p)

print(f"{len(feeders)} feeders placed")

# --- Surround with FEED mode ---
orders = surround(feeders, queen.location, grid, elevs,
                  mode=SurroundMode.FEED, max_ring=2)

print(f"Assigned {len(orders)} pieces")
for p in feeders:
    if p.id in orders:
        p.instructions = orders[p.id]
        print(f"  {p.name} ({p.piece_type.name}): {len(p.instructions.rules)} instructions")

# --- Render with plan overlay ---
#PieceBoardPlan(board, num_turns=8)


this is so perfect. Maybe we could add a 
```
@patch
def facing_bar(self: DiagramGlyphs, cx, cy, facing: int,
               length=None, offset=None, stroke=None,
               stroke_width=None, opacity=None) -> str:
    """Short flat bar on the 'front' side of the piece — football blocking style."""
    import math
    length = length or self.size * 0.7
    offset = offset or self.size * 0.55
    stroke = stroke or self.color
    stroke_width = stroke_width or max(2, self.size * 0.15)
    opacity = opacity if opacity is not None else self.opacity

    # Facing angle: direction 0 = SE in your hex layout
    # Adjust this base angle to match your HexPosition.directions() order
    angle = math.radians(60 * facing)  # tune to your coord system

    # Center of the bar, pushed outward along facing
    bx = cx + offset * math.cos(angle)
    by = cy + offset * math.sin(angle)




    # Perpendicular endpoints
    perp = angle + math.pi / 2
    half = length / 2
    x1, y1 = bx + half * math.cos(perp), by + half * math.sin(perp)
    x2, y2 = bx - half * math.cos(perp), by - half * math.sin(perp)

    return (f'<line x1="{x1:.1f}" y1="{y1:.1f}" '
            f'x2="{x2:.1f}" y2="{y2:.1f}" '
            f'stroke="{stroke}" stroke-width="{stroke_width:.1f}" '
            f'stroke-linecap="round" opacity="{opacity}"/>\n')
            ```
            at the end to see where the pieces are facing

In [ ]:
@patch
def _facing_overlay(self: PieceBoard, num_turns: int = 8) -> str:
    """Facing bars at each piece's final simulated position."""
    svg = ""
    for piece in self.pieces:
        if not piece.instructions.rules:
            continue
        steps = piece.simulate(self.grid, self.elevations,
                               num_turns=num_turns,
                               countries=self.countries)
        if not steps:
            continue
        
        last = steps[-1]
        c = self.grid.hexes[last.hex_idx].center
        color = piece.flag.primary if piece.flag else "#333"
        g = DiagramGlyphs(color=color, size=self.grid.radius * 0.45)
        svg += g.facing_bar(c.x, c.y, last.facing)
    
    return svg


@patch
def render_with_facing(self: PieceBoard, num_turns: int = 8) -> str:
    """Board render with plan overlay + final facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("pieces",  self._pieces_overlay())
    self.grid.builder.adjust("plan",    self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing",  self._facing_overlay(num_turns))
    
    xs = [h.center.x for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    ys = [h.center.y for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()


In [ ]:
@patch
def _facing_overlay(self: PieceBoard, num_turns: int = 8) -> str:
    """Facing bars at each piece's final simulated position."""
    svg = ""
    for piece in self.pieces:
        if not piece.instructions.rules:
            continue
        steps = piece.simulate(self.grid, self.elevations,
                               num_turns=num_turns,
                               countries=self.countries)
        if not steps:
            continue

        last = steps[-1]
        c = self.grid.hexes[last.hex_idx].center
        color = piece.flag.primary if piece.flag else "#fff"
        
        # Dark outline pass first, then bright bar on top
        g_outline = DiagramGlyphs(color="#222222",
                                  size=self.grid.radius * 0.55,
                                  stroke_width=4.5, opacity=0.9)
        g_bar = DiagramGlyphs(color="#FFFFFF",
                              size=self.grid.radius * 0.55,
                              stroke_width=2.5, opacity=1.0)
        
        svg += g_outline.facing_bar(c.x, c.y, last.facing)
        svg += g_bar.facing_bar(c.x, c.y, last.facing)

    return svg


In [ ]:
@patch
def render_with_facing(self: PieceBoard, num_turns: int = 8) -> str:
    """Board render with plan overlay + final facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("pieces",  self._pieces_overlay())
    self.grid.builder.adjust("plan",    self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing",  self._facing_overlay(num_turns))
    
    xs = [h.center.x for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    ys = [h.center.y for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()



In [ ]:
#show(NotStr(board.render_with_facing(num_turns=8)))

I can't really see the bars. Are they under? should we use a different color?

so they should extend further towards the edge of the hex we also want thing rotated to face the queen

In [ ]:
@patch
def facing_bar(self: DiagramGlyphs, cx, cy, facing: int,
               length=None, offset=None, stroke=None,
               stroke_width=None, opacity=None) -> str:
    """Short flat bar on the 'front' side of the piece — football blocking style."""
    import math
    length       = length       or self.size * 1.2          # wider bar
    offset       = offset       or self.size * 1.1          # pushed toward hex edge
    stroke       = stroke       or self.color
    stroke_width = stroke_width or max(2, self.size * 0.15)
    opacity      = opacity if opacity is not None else self.opacity

    # SW=0,W=1,NW=2,NE=3,E=4,SE=5 → pixel angles 120,180,240,300,0,60
    angle = math.radians(60 * facing + 120)

    bx = cx + offset * math.cos(angle)
    by = cy + offset * math.sin(angle)

    perp = angle + math.pi / 2
    half = length / 2
    x1, y1 = bx + half * math.cos(perp), by + half * math.sin(perp)
    x2, y2 = bx - half * math.cos(perp), by - half * math.sin(perp)

    return (f'<line x1="{x1:.1f}" y1="{y1:.1f}" '
            f'x2="{x2:.1f}" y2="{y2:.1f}" '
            f'stroke="{stroke}" stroke-width="{stroke_width:.1f}" '
            f'stroke-linecap="round" opacity="{opacity}"/>\n')


@patch
def _facing_overlay(self: PieceBoard, num_turns: int = 8) -> str:
    """Facing bars at each piece's final simulated position."""
    svg = ""
    r = self.grid.radius
    for piece in self.pieces:
        if not piece.instructions.rules:
            continue
        steps = piece.simulate(self.grid, self.elevations,
                               num_turns=num_turns,
                               countries=self.countries)
        if not steps:
            continue

        last = steps[-1]
        c = self.grid.hexes[last.hex_idx].center

        # Size drives offset/length via the multipliers above
        g_outline = DiagramGlyphs(color="#111111", size=r * 0.7,
                                  stroke_width=5.0, opacity=0.85)
        g_bar     = DiagramGlyphs(color="#FFFFFF",  size=r * 0.7,
                                  stroke_width=2.8, opacity=1.0)

        svg += g_outline.facing_bar(c.x, c.y, last.facing)
        svg += g_bar.facing_bar(c.x, c.y, last.facing)

    return svg


In [ ]:
#show(NotStr(board.render_with_facing(num_turns=8)))

So I think we are ready for our sightOverlay. we want to see which hexes are covered by a list of pieces. perhaps the overlay could be a dotted one similar to food, but with the kingdom color

In [ ]:
@patch
def hexes_in_sight(self: Piece, grid: HexGrid, elevations: np.ndarray,
                   elevation_mult: float = 0.005,
                   facing_only: bool = False) -> list[int]:
    """All hex indices visible from this piece's location."""
    if self.location is None or self.location < 0:
        return []
    
    elev = max(0, elevations[self.location])
    effective_sight = int(self.sight + elev * elevation_mult)
    
    if facing_only:
        facing_dir = HexPosition.directions()[self.facing % 6]
        hex_positions = field_of_view(HexPosition.origin(), facing_dir, effective_sight)
        return [
            idx for hp in hex_positions
            if (idx := grid.hexposition_to_index(hp, self.location)) >= 0
        ]
    else:
        return grid.indices_in_range(self.location, effective_sight)


In [ ]:



@patch
def sight_overlay(self: PieceBoard, pieces: list[Piece],
                  elevation_mult: float = 0.005,
                  opacity: float = 0.22,
                  facing_only: bool = False,
                  allowed_rotations: int = 1) -> str:
    """Dotted sight-range overlay, one color per kingdom."""
    svg = ""
    r = self.grid.radius
    dot_r   = max(1.5, r * 0.10)
    spacing = max(5.0, r * 0.32)

    # Group by kingdom color
    color_to_hexes: dict[str, set[int]] = {}
    for piece in pieces:
        color = piece.flag.primary if piece.flag else "#888"
        visible = piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=elevation_mult,
            facing_only=facing_only,
            allowed_rotations=allowed_rotations
        )
        color_to_hexes.setdefault(color, set()).update(visible)

    for color, hex_indices in color_to_hexes.items():
        pat_id = f"sight_{color.replace('#','')}"
        # Register dot pattern as a def
        pat_svg = (
            f'<pattern id="{pat_id}" x="0" y="0" '
            f'width="{spacing:.1f}" height="{spacing:.1f}" '
            f'patternUnits="userSpaceOnUse">'
            f'<circle cx="{spacing/2:.1f}" cy="{spacing/2:.1f}" '
            f'r="{dot_r:.1f}" fill="{color}"/>'
            f'</pattern>'
        )
        self.grid.builder.add_definition(
            SVGDef("", pat_id, pat_svg, raw=True)
        )

        for idx in sorted(hex_indices):
            if idx < 0 or idx >= len(self.grid.hexes):
                continue
            h = self.grid.hexes[idx]
            pts = " ".join(f"{v.x},{v.y}" for v in h.v)
            svg += (
                f'<polygon points="{pts}" '
                f'fill="url(#{pat_id})" opacity="{opacity:.2f}" '
                f'stroke="{color}" stroke-width="0.6" '
                f'stroke-dasharray="3,2" stroke-opacity="0.4"/>\n'
            )
    return svg


@patch
def render_with_sight(self: PieceBoard, pieces: list[Piece],
                      num_turns: int = 8,
                      facing_only: bool = False) -> str:
    """Board with sight overlay + movement plans + facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("sight",  self.sight_overlay(pieces, facing_only=facing_only))
    self.grid.builder.adjust("pieces", self._pieces_overlay())
    self.grid.builder.adjust("plan",   self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing", self._facing_overlay(num_turns))

    xs = [h.center.x for h in self.grid.hexes]
    ys = [h.center.y for h in self.grid.hexes]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()


 think we need to work on field of view which seems to go wider that 120. in ring 1 it is fine. but in ring 2 it should go straing out from its edges but it bends

These wound up too suttble. I wonder if darkPrimary or baseComp or perhaps larger dots makes more sense.

In [ ]:
@patch
def sight_overlay(self: PieceBoard, pieces: list[Piece],
                  elevation_mult: float = 0.005,
                  opacity: float = 0.35,          # up from 0.22
                  facing_only: bool = False,
                  allowed_rotations: int = 1,
                  color_attr: str = "darkPrimary"  # "primary" | "darkPrimary" | "baseComp"
                  ) -> str:
    """Dotted sight-range overlay, one color per kingdom."""
    svg = ""
    r = self.grid.radius
    dot_r   = max(2.5, r * 0.18)   # up from 0.10
    spacing = max(5.0, r * 0.32)

    color_to_hexes: dict[str, set[int]] = {}
    for piece in pieces:
        color = getattr(piece.flag, color_attr, "#888") if piece.flag else "#888"
        visible = piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=elevation_mult,
            facing_only=facing_only
        )
        color_to_hexes.setdefault(color, set()).update(visible)

    for color, hex_indices in color_to_hexes.items():
        pat_id = f"sight_{color.replace('#','')}"
        pat_svg = (
            f'<pattern id="{pat_id}" x="0" y="0" '
            f'width="{spacing:.1f}" height="{spacing:.1f}" '
            f'patternUnits="userSpaceOnUse">'
            f'<circle cx="{spacing/2:.1f}" cy="{spacing/2:.1f}" '
            f'r="{dot_r:.1f}" fill="{color}"/>'
            f'</pattern>'
        )
        self.grid.builder.add_definition(
            SVGDef("", pat_id, pat_svg, raw=True)
        )

        for idx in sorted(hex_indices):
            if idx < 0 or idx >= len(self.grid.hexes):
                continue
            h = self.grid.hexes[idx]
            pts = " ".join(f"{v.x},{v.y}" for v in h.v)
            svg += (
                f'<polygon points="{pts}" '
                f'fill="url(#{pat_id})" opacity="{opacity:.2f}" '
                f'stroke="{color}" stroke-width="0.8" '
                f'stroke-dasharray="3,2" stroke-opacity="0.5"/>\n'
            )
    return svg


Lets do a demo with one piece so I can see the cone

# darkPrimary — recommended
show(NotStr(board.render_with_sight(feeders + [queen], num_turns=8)))

In [ ]:
def field_of_view(origin: HexPosition, facing: HexPosition, max_distance: int) -> list[HexPosition]:
    """Get all hexes within a 120° field of view (±60° from facing) up to max_distance.
    
    Walks the cone arc per ring — no per-hex tests needed.
    """
    left_edge = facing.rotate(1)     # start of arc (left side)
    step1 = facing.rotate(-1)        # walk from left toward center
    step2 = facing.rotate(-2)        # walk from center toward right
    
    results = []
    for r in range(1, max_distance + 1):
        current = origin + r * left_edge
        for _ in range(r):
            results.append(current)
            current = current + step1
        for _ in range(r):
            results.append(current)
            current = current + step2
        results.append(current)      # final right-edge hex
    return results


# --- Fresh board ---
board = PieceBoard.hilly(rings=7, radius=20, seed=42)
grid  = board.grid
elevs = board.elevations

# Find a nice land hex near center
mid = grid.middle
scout_idx = None
for dist in range(0, 6):
    for hp in HexPosition.origin().ring(dist):
        idx = grid.hexposition_to_index(hp, mid)
        if idx >= 0 and elevs[idx] > 0:
            scout_idx = idx
            break
    if scout_idx is not None:
        break

# Single scout, facing direction 3 (NE) so cone points up-right
scout = board.add_piece(scout_idx, PieceType.KNIGHT, facing=3, country_id=1,
                        instructions=InstructionList([Instruction.PAUSE.value], patrol=True))
scout.sight = 4   # decent range

print(f"Scout at hex {scout_idx} (elev={elevs[scout_idx]:.0f}), facing=3 (NE)")
print(f"Effective sight: {scout.effective_sight(elevs):.2f}")

# Cone only — shows the 120° wedge
show(NotStr(board.render_with_sight(
    [scout],
    num_turns=1,
    facing_only=True   # ← cone mode
)))


So I am wondering about another matching algorith. we want to cover a path so that as a piece walks along it they would be fed. so we need to put pieces on the best yields "outside of the path". and then use a similar bipartite graph to match them. I think these field of view algorithms will come in handy for getting covverage.

In [ ]:
def supply_line(path: list[int], feeders: list[Piece],
                grid: HexGrid, elevations: np.ndarray,
                food_tiers: np.ndarray,
                min_tier: int = 2,
                max_offset: int = 2,
                countries: np.ndarray = None,
                elevation_mult: float = 0.005
                ) -> dict[str, InstructionList]:
    """Station feeders on fertile tiles along a march path.
    
    Args:
        path: Hex indices of the march route
        feeders: Available pieces to assign as suppliers
        grid: HexGrid
        elevations: Elevation array
        food_tiers: Per-hex food tier (0..n_tiers)
        min_tier: Minimum food tier for a candidate position
        max_offset: How many rings off-path to search
        countries: Country passability array
        elevation_mult: Sight bonus per elevation unit
    
    Returns:
        Dict mapping piece.id → InstructionList (path + face + harvest/give loop)
    """
    path_set = set(path)
    
    # --- Phase 1: Generate scored candidates ---
    # candidate = (hex_idx, facing_toward_path, food_tier, covered_path_hexes)
    candidates = {}  # idx -> (facing, tier, covered_set)
    
    for path_hex in path:
        for ring in range(1, max_offset + 1):
            for hp in HexPosition.origin().ring(ring):
                idx = grid.hexposition_to_index(hp, path_hex)
                if idx < 0 or idx in grid.invalidRegion:
                    continue
                if elevations[idx] <= 0:
                    continue
                if idx in path_set:
                    continue
                if int(food_tiers[idx]) < min_tier:
                    continue
                if countries is not None and countries[idx] < 0:
                    continue
                
                # Face toward nearest path hex
                facing = _facing_toward(grid, idx, path_hex)
                facing_dir = HexPosition.directions()[facing]
                
                # What path hexes does this candidate's GIVE cone cover?
                elev = max(0, elevations[idx])
                eff_sight = 3 + elev * elevation_mult  # default sight for supply calc
                fov = set(field_of_view(HexPosition.origin(), facing_dir, int(eff_sight)))
                
                covered = set()
                for ph in path:
                    rel = grid.index_to_hexposition(ph, idx)
                    if rel in fov:
                        covered.add(ph)
                
                if not covered:
                    continue
                
                tier = int(food_tiers[idx])
                
                # Keep best coverage per candidate hex
                if idx not in candidates or len(covered) > len(candidates[idx][2]):
                    candidates[idx] = (facing, tier, covered)
    
    if not candidates:
        return {}
    
    # --- Phase 2: Greedy set cover ---
    uncovered = set(path)
    selected = []  # (idx, facing, tier)
    remaining = dict(candidates)
    
    while uncovered and remaining:
        # Pick candidate with best score: coverage × tier
        best_idx = max(remaining, key=lambda i: (
            len(remaining[i][2] & uncovered) * remaining[i][1]
        ))
        facing, tier, covered = remaining.pop(best_idx)
        
        newly_covered = covered & uncovered
        if not newly_covered:
            continue
        
        selected.append((best_idx, facing, tier))
        uncovered -= newly_covered
    
    if not selected:
        return {}
    
    # --- Phase 3: Hungarian matching ---
    n_feeders = len(feeders)
    n_positions = len(selected)
    
    INF = 1e9
    cost = np.full((n_feeders, n_positions), INF)
    
    for i, piece in enumerate(feeders):
        if piece.location is None or piece.location < 0:
            continue
        for j, (pos_idx, _, _) in enumerate(selected):
            p = piece.pathfind(pos_idx, grid, elevations, countries)
            if not p:
                continue
            c = sum(_move_cost(elevations, p[k], p[k+1]) for k in range(len(p)-1))
            cost[i, j] = c
    
    row_ind, col_ind = linear_sum_assignment(cost)
    
    # --- Phase 4: Build instructions ---
    assignments = {}
    for i, j in zip(row_ind, col_ind):
        if cost[i, j] >= INF:
            continue
        
        piece = feeders[i]
        pos_idx, facing, _ = selected[j]
        
        # Path to position
        p = piece.pathfind(pos_idx, grid, elevations, countries)
        if len(p) < 2:
            continue
        rules = InstructionList.path_to_rules(p, grid, start_facing=piece.facing)
        
        # Rotate to face path
        path_end_facing = _facing_toward(grid, p[-2], p[-1]) if len(p) >= 2 else piece.facing
        diff = (facing - path_end_facing) % 6
        if diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))
        
        # Patrol loop: harvest then give
        rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
        
        assignments[piece.id] = InstructionList(rules, cursor=0, patrol=True)
    
    return assignments


can you build a demo. for gameboards maybe just assume elevation determines food yield for our example

In [ ]:
@patch
def pathfind_to_rules(self: Piece, target: int, grid: HexGrid,
                      elevations: np.ndarray,
                      countries: np.ndarray = None) -> InstructionList:
    """Find path to target and convert to movement instructions."""
    path = self.pathfind(target, grid, elevations, countries)
    if len(path) < 2:
        return InstructionList([])
    
    rules = InstructionList.path_to_rules(path, grid, start_facing=self.facing)
    return InstructionList(rules, cursor=0, patrol=False)


@patch
def pathfind_to_food(self: Piece, grid: HexGrid,
                     elevations: np.ndarray, food_tiers: np.ndarray,
                     min_tier: int = 3,
                     countries: np.ndarray = None) -> InstructionList:
    """Find nearest tile with food >= min_tier and path there + harvest."""
    start = self.location
    if start is None or start < 0:
        return InstructionList([])
    
    pq = [(0.0, start)]
    visited = set()
    parent = {start: None}
    
    while pq:
        cost, current = heapq.heappop(pq)
        
        if current in visited:
            continue
        visited.add(current)
        
        if current != start and int(food_tiers[current]) >= min_tier:
            path = []
            node = current
            while node is not None:
                path.append(node)
                node = parent[node]
            path.reverse()
            
            rules = InstructionList.path_to_rules(path, grid, start_facing=self.facing)
            rules.append(Instruction.HARVEST.value)
            return InstructionList(rules, cursor=0, patrol=False)
        
        for neighbor in grid.neighborsOf(current):
            if neighbor in visited or neighbor in grid.invalidRegion:
                continue
            if elevations[neighbor] <= 0:
                continue
            if countries is not None and countries[neighbor] < 0:
                continue
            
            edge_cost = _move_cost(elevations, current, neighbor)
            if neighbor not in parent:
                parent[neighbor] = current
                heapq.heappush(pq, (cost + edge_cost, neighbor))
    
    return InstructionList([])


In [ ]:
# --- Board setup ---
board = PieceBoard.hilly(rings=8, radius=18, seed=42)
grid  = board.grid
elevs = board.elevations
mid   = grid.middle

# Fake food tiers from elevation
food_tiers = np.zeros(len(elevs), dtype=int)
for i, e in enumerate(elevs):
    if e <= 0:      food_tiers[i] = 0
    elif e < 80:    food_tiers[i] = 6
    elif e < 140:   food_tiers[i] = 5
    elif e < 200:   food_tiers[i] = 3
    elif e < 280:   food_tiers[i] = 1
    else:           food_tiers[i] = 0

# --- Find start and target on the NORTH side (above river) ---
def find_land(offsets):
    for hp in offsets:
        idx = grid.hexposition_to_index(hp, mid)
        if idx >= 0 and elevs[idx] > 0:
            return idx
    return None

start_idx  = find_land([HexPosition(-5, -3, 8), HexPosition(-4, -3, 7), HexPosition(-3, -2, 5)])
target_idx = find_land([HexPosition(5, -3, -2), HexPosition(4, -3, -1), HexPosition(3, -2, -1)])

print(f"Start: hex {start_idx} (elev={elevs[start_idx]:.0f})")
print(f"Target: hex {target_idx} (elev={elevs[target_idx]:.0f})")

# --- Queen ---
queen = board.add_piece(start_idx, PieceType.QUEEN, facing=4, country_id=1,
                        instructions=InstructionList([]))
queen.food = 3.0
march_path = queen.pathfind(target_idx, grid, elevs)
queen.instructions = queen.pathfind_to_rules(target_idx, grid, elevs)
print(f"March path: {len(march_path)} hexes, {len(queen.instructions.rules)} instructions")

# --- Scatter feeders on the NORTH side ---
feeder_offsets = [
    (-4, -1), (-3, -3), (-1, -2), (0, -3), (1, -1),
    (2, -3), (3, -1), (-2, -1), (4, -2), (-1, -4)
]
feeders = []
for dq, dr in feeder_offsets:
    hp = HexPosition(dq, dr, -dq - dr)
    idx = grid.hexposition_to_index(hp, mid)
    if idx < 0 or elevs[idx] <= 0:
        continue
    p = board.add_piece(idx, PieceType.PAWN, facing=random.randint(0, 5), country_id=1,
                        instructions=InstructionList([]))
    p.food = 5.0
    p.sight = 3
    feeders.append(p)

print(f"{len(feeders)} feeders available")

# --- Supply line ---
orders = supply_line(march_path, feeders, grid, elevs, food_tiers,
                     min_tier=3, max_offset=2)

print(f"\nAssigned {len(orders)} feeders:")
for p in feeders:
    if p.id in orders:
        p.instructions = orders[p.id]
        print(f"  {p.name}: {len(p.instructions.rules)} instrs")

# --- Render ---
assigned = [p for p in feeders if p.id in orders]
show(NotStr(board.render_with_sight(
    assigned + [queen],
    num_turns=12,
    facing_only=True
)))


Are there some good . Patrol Routes (Gosper Curve)

Purpose: Efficiently cover area without backtracking.

Algorithm: Space-filling curve generation

Concept:

    Gosper curve is hex-native space-filling curve
    Covers region with minimal repeated hexes
    Natural for patrol, scouting, area denial

Implementation:
Copied!

def patrol_route(anchor: int, radius: int, grid: HexGrid,
                 curve_order=2) -> list[int]:
    """
    Generate Gosper curve patrol route.
    
    1. Build Gosper curve of order N around anchor
    2. Filter to hexes within radius
    3. Convert curve to hex indices
    4. Return as InstructionList (path_to_rules)
    """

Patrol Behaviors:

    SCOUT — high sight pieces on wide patrol (order 3-4)
    PERIMETER — patrol kingdom border (anchor = capital)
    SWEEP — search for enemy in region

Strategic Uses:

    Early game exploration
    Border security
    Resource scouting

we could work on?

My quesion is about if we had a set of things that we wanted to patrol. should we assign them first to space and then patrol. is something optimal this way?

In [ ]:
!cat ../../HexMagic/voronoi.py

I have some voronoi already built. Can we leverage some of this? is from sklearn.cluster import KMeans going to help. or is it we do voronoi type stuff from a piece. Thoughts about using this for our "explore/patrol" the world

I like the idea of weighted expansion. we have colonies where there could be vital strategic stuff (or planning we just want to expand our borders). I think the vornoi expand in conjuction with optimizy might give us could places to set out outposts and how to patrol them.

I kind of want to go down the n exapnd route. the idea is that I have n groups and I want them to move to k nearby locations. we need to find the locations and then assign the groups. so this is really 1-3. then a separate algrithm would be how to defend and existing colonoy

Do you think you could built all three phases? 

In [ ]:
#| export

def expansion_zones(colonies: list[int], grid: HexGrid, elevations: np.ndarray,
                    countries: np.ndarray = None) -> tuple[np.ndarray, np.ndarray]:
    """Multi-seed Dijkstra — grow influence zones from colony hexes.
    
    Returns:
        zone_of: array[n_hexes] → colony index (-1 = unclaimed/ocean)
        cost_of: array[n_hexes] → travel cost from owning colony
    """
    n = len(elevations)
    zone_of = np.full(n, -1, dtype=int)
    cost_of = np.full(n, np.inf)
    
    pq = []
    for ci, colony_idx in enumerate(colonies):
        if colony_idx < 0 or elevations[colony_idx] <= 0:
            continue
        zone_of[colony_idx] = ci
        cost_of[colony_idx] = 0.0
        heapq.heappush(pq, (0.0, colony_idx, ci))
    
    while pq:
        cost, current, owner = heapq.heappop(pq)
        
        # Skip if already claimed by someone cheaper
        if cost > cost_of[current]:
            continue
        
        for neighbor in grid.neighborsOf(current):
            if neighbor in grid.invalidRegion:
                continue
            if elevations[neighbor] <= 0:
                continue
            if countries is not None and countries[neighbor] < 0:
                continue
            
            edge_cost = _move_cost(elevations, current, neighbor)
            new_cost = cost + edge_cost
            
            if new_cost < cost_of[neighbor]:
                cost_of[neighbor] = new_cost
                zone_of[neighbor] = owner
                heapq.heappush(pq, (new_cost, neighbor, owner))
    
    return zone_of, cost_of


def score_expansion_sites(zone_of: np.ndarray, cost_of: np.ndarray,
                          grid: HexGrid, elevations: np.ndarray,
                          food_tiers: np.ndarray,
                          n_candidates: int = 5,
                          min_cost: float = 3.0,
                          food_w: float = 1.0,
                          elev_w: float = 0.3,
                          frontier_w: float = 2.0,
                          choke_w: float = 1.5,
                          dist_w: float = 0.1
                          ) -> list[tuple[int, float, int]]:
    """Score hexes as expansion/outpost candidates.
    
    Args:
        zone_of, cost_of: from expansion_zones()
        min_cost: minimum distance from parent colony (don't build next door)
        *_w: scoring weights
    
    Returns:
        List of (hex_idx, score, zone_owner) sorted by score descending
    """
    n = len(elevations)
    scores = []
    
    for i in range(n):
        if zone_of[i] < 0 or elevations[i] <= 0:
            continue
        if cost_of[i] < min_cost:
            continue  # too close to parent colony
        
        tier = int(food_tiers[i]) if i < len(food_tiers) else 0
        elev = max(0, elevations[i])
        
        # Frontier proximity: how close to a zone boundary?
        frontier_score = 0.0
        n_blocked = 0
        for nb in grid.neighborsOf(i):
            if nb in grid.invalidRegion or elevations[nb] <= 0:
                n_blocked += 1
                continue
            if zone_of[nb] >= 0 and zone_of[nb] != zone_of[i]:
                frontier_score = 1.0  # on the border
                break
        
        # Chokepoint: count impassable neighbors (water/mountain/invalid)
        total_nb = len(grid.neighborsOf(i))
        choke_factor = n_blocked / max(1, total_nb)  # 0 = open, 1 = peninsula
        
        score = (tier * food_w
                 + elev * 0.01 * elev_w
                 + frontier_score * frontier_w
                 + choke_factor * choke_w
                 - cost_of[i] * dist_w)
        
        scores.append((i, score, zone_of[i]))
    
    scores.sort(key=lambda x: -x[1])
    
    # Pick top candidates, but spread across zones and space them out
    selected = []
    used_zones = {}  # zone → count
    selected_set = set()
    min_spacing = 4  # minimum hex distance between outposts
    
    for idx, score, zone in scores:
        if len(selected) >= n_candidates:
            break
        
        # Check spacing from already-selected sites
        pos_i = grid.index_to_hexposition(idx)
        too_close = False
        for sel_idx, _, _ in selected:
            pos_j = grid.index_to_hexposition(sel_idx)
            if pos_i.distance(pos_j) < min_spacing:
                too_close = True
                break
        if too_close:
            continue
        
        selected.append((idx, score, zone))
    
    return selected


def assign_groups(groups: list[list[Piece]], targets: list[tuple[int, float, int]],
                  grid: HexGrid, elevations: np.ndarray,
                  countries: np.ndarray = None
                  ) -> dict[str, InstructionList]:
    """Hungarian match piece groups → expansion sites.
    
    Cost for a group = max pathfind cost of any member (bottleneck).
    
    Args:
        groups: list of piece-lists (each group moves together)
        targets: from score_expansion_sites — (hex_idx, score, zone)
    
    Returns:
        Dict mapping piece.id → InstructionList for every piece in matched groups
    """
    n_groups = len(groups)
    n_targets = len(targets)
    if n_groups == 0 or n_targets == 0:
        return {}
    
    INF = 1e9
    cost = np.full((n_groups, n_targets), INF)
    
    # Cache paths: (group_idx, target_idx) → {piece_id: path}
    path_cache = {}
    
    for gi, group in enumerate(groups):
        for tj, (target_idx, _, _) in enumerate(targets):
            group_paths = {}
            max_cost = 0.0
            all_reachable = True
            
            for piece in group:
                if piece.location is None or piece.location < 0:
                    all_reachable = False
                    break
                path = piece.pathfind(target_idx, grid, elevations, countries)
                if not path:
                    all_reachable = False
                    break
                
                pc = sum(_move_cost(elevations, path[k], path[k+1])
                         for k in range(len(path) - 1))
                max_cost = max(max_cost, pc)
                group_paths[piece.id] = path
            
            if all_reachable:
                cost[gi, tj] = max_cost
                path_cache[(gi, tj)] = group_paths
    
    # Hungarian matching
    row_ind, col_ind = linear_sum_assignment(cost)
    
    # Build instructions for each matched piece
    assignments = {}
    for gi, tj in zip(row_ind, col_ind):
        if cost[gi, tj] >= INF:
            continue
        
        group = groups[gi]
        group_paths = path_cache.get((gi, tj), {})
        
        for piece in group:
            path = group_paths.get(piece.id)
            if not path or len(path) < 2:
                continue
            
            rules = InstructionList.path_to_rules(path, grid, start_facing=piece.facing)
            assignments[piece.id] = InstructionList(rules, cursor=0, patrol=False)
    
    return assignments


def expand(colonies: list[int], groups: list[list[Piece]],
           grid: HexGrid, elevations: np.ndarray,
           food_tiers: np.ndarray,
           n_outposts: int = None,
           countries: np.ndarray = None,
           **scoring_kw
           ) -> tuple[dict[str, InstructionList], list[tuple[int, float, int]]]:
    """Full expansion pipeline: zone → score → assign.
    
    Args:
        colonies: hex indices of existing settlements
        groups: piece groups to send out
        n_outposts: how many sites to find (default = len(groups))
        **scoring_kw: passed to score_expansion_sites
    
    Returns:
        (assignments, sites) — piece instructions + chosen outpost locations
    """
    if n_outposts is None:
        n_outposts = len(groups)
    
    # Phase 1: zones
    zone_of, cost_of = expansion_zones(colonies, grid, elevations, countries)
    
    # Phase 2: score sites
    sites = score_expansion_sites(
        zone_of, cost_of, grid, elevations, food_tiers,
        n_candidates=n_outposts, **scoring_kw
    )
    
    # Phase 3: assign groups
    assignments = assign_groups(groups, sites, grid, elevations, countries)
    
    return assignments, sites


In [ ]:
myBuilder = myTerr.hexGrid.builder
myBuilder.adjust("capitals",myBoard.settlementOverlay())
myBuilder.show()

In [ ]:
myTerr.hexGrid.radius

In [ ]:
myTerr.hexGrid.adjustRadius(30)

In [ ]:
for country in myBoard.kingdoms:
    print(country.countryName)

In [ ]:
for pType in [PieceType.PAWN,PieceType.PAWN,PieceType.PAWN,PieceType.PAWN,PieceType.PAWN,PieceType.QUEEN]:
    pie

In [ ]:
show(NotStr(myBoard.render_with_sight(
        queenGuard.pieces,
        num_turns=12,
        facing_only=True
    )))

In [ ]:
myQueen.location

In [ ]:
dest = myQueen.location - 8

queenPath = myQueen.pathfind(target=dest,grid=myGrid,elevations=myTerr.elevations)

myQueen.rules = queenPath.to_rules(grid=myGrid)

show(myQueen)

In [ ]:
queenGuard.surround(target=dest, grid=myGrid, elevations=myTerr.elevations)
myQueen.instructions = myQueen.pathfind_to_rules(target=dest, grid=myGrid, elevations=myTerr.elevations)
myQueen.instructions.rules.append(Instruction.PAUSE)

In [ ]:
## Another squad

Is zoomed creating the right c2f?

In [ ]:
OverlayContext.from_terrain??

can you write the proper from_terrain?